# Modelo Mensual — OLS-Augmented (v7 + OLS_Pred feature)
## Ensemble Segmentado con OLS como Variable

**Diferencia vs Modelo_Mensual_FinalV1**: se agrega `OLS_Pred` como feature adicional
en los modelos multi-horizonte. OLS pre-computado por rolling origin (sin fuga).

**Nuevos experimentos**: `LGB_MultiH_OLS`, `RF_MultiH_OLS`, `LGB_MultiH_LC_OLS`, `RF_MultiH_LC_OLS`


## 1. Imports y Configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, copy

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.model_selection import TimeSeriesSplit

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings('ignore')

HORIZONS        = [1, 3, 6, 9, 12, 18, 24, 30, 36, 42, 48, 54, 60]
H_FOCUS         = [36, 48, 60]
LGB_TRIALS      = 30
XGB_TRIALS      = 15
CB_TRIALS       = 15
RF_TRIALS       = 15
MLP_TRIALS      = 20
MAX_LEAVES      = 20
TUNE_H          = 36
EXPORT_DIR      = 'outputs_best'
os.makedirs(EXPORT_DIR, exist_ok=True)

ROLLING_ORIGINS = pd.date_range('2017-01-01', '2022-01-01', freq='6MS').tolist()

EXCLUDE_MINES = {
    'haldeman', 'altos de punitaqui', 'tres valles',
    'pampa camarones', 'michilla', 'quebrada blanca', 'spence',
    'cerro negro'
}

# COVID-19 pandemic: exogenous shock indicators
# Is_Pandemic_Orig  (in crear_features): 1 if origin date is in pandemic window
# Is_Pandemic_Target (in ML loop): 1 if target date is in pandemic window
# KEY FIX: origins 2021-01 and 2021-07 have WR < 50% because model mistakes
# pandemic-depressed baseline as a structural decline rather than a temporary shock.
PANDEMIC_START = pd.Timestamp('2020-03-01')
PANDEMIC_END   = pd.Timestamp('2021-12-31')

print('Configuración cargada')
print(f'Orígenes: {[o.strftime("%Y-%m") for o in ROLLING_ORIGINS]}')
print(f'Horizontes foco: H+{H_FOCUS}m')
print(f'Excluidas: {sorted(EXCLUDE_MINES)}')
print(f'Pandemic window: {PANDEMIC_START.date()} → {PANDEMIC_END.date()}')

Configuración cargada
Orígenes: ['2017-01', '2017-07', '2018-01', '2018-07', '2019-01', '2019-07', '2020-01', '2020-07', '2021-01', '2021-07', '2022-01']
Horizontes foco: H+[36, 48, 60]m
Excluidas: ['altos de punitaqui', 'cerro negro', 'haldeman', 'michilla', 'pampa camarones', 'quebrada blanca', 'spence', 'tres valles']
Pandemic window: 2020-03-01 → 2021-12-31


## 2. Wrappers de Modelos y Optuna

In [2]:
class LGBWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, n_estimators=500, learning_rate=0.03, num_leaves=15,
                 min_child_samples=7, reg_alpha=0.1, reg_lambda=1.0,
                 objective='regression'):
        self.n_estimators=n_estimators; self.learning_rate=learning_rate
        self.num_leaves=num_leaves; self.min_child_samples=min_child_samples
        self.reg_alpha=reg_alpha; self.reg_lambda=reg_lambda
        self.objective=objective
    def fit(self, X, y, **kw):
        self.model_ = lgb.LGBMRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            num_leaves=self.num_leaves, min_child_samples=self.min_child_samples,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            objective=self.objective,
            random_state=42, verbose=-1)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self,'model_'); return self.model_.predict(X)

class XGBWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, n_estimators=400, learning_rate=0.03, max_depth=4,
                 min_child_weight=5, reg_alpha=0.1, reg_lambda=1.0, subsample=0.8):
        self.n_estimators=n_estimators; self.learning_rate=learning_rate
        self.max_depth=max_depth; self.min_child_weight=min_child_weight
        self.reg_alpha=reg_alpha; self.reg_lambda=reg_lambda; self.subsample=subsample
    def fit(self, X, y, **kw):
        self.model_ = xgb.XGBRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            max_depth=self.max_depth, min_child_weight=self.min_child_weight,
            reg_alpha=self.reg_alpha, reg_lambda=self.reg_lambda,
            subsample=self.subsample, random_state=42, verbosity=0, n_jobs=2)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self, 'model_'); return self.model_.predict(X)

class CBWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, iterations=400, learning_rate=0.03, depth=4,
                 l2_leaf_reg=3.0, min_data_in_leaf=5):
        self.iterations=iterations; self.learning_rate=learning_rate
        self.depth=depth; self.l2_leaf_reg=l2_leaf_reg
        self.min_data_in_leaf=min_data_in_leaf
    def fit(self, X, y, **kw):
        self.model_ = CatBoostRegressor(
            iterations=self.iterations, learning_rate=self.learning_rate,
            depth=self.depth, l2_leaf_reg=self.l2_leaf_reg,
            min_data_in_leaf=self.min_data_in_leaf,
            random_seed=42, verbose=False)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self, 'model_'); return self.model_.predict(X)

class RFWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, n_estimators=200, max_depth=8, min_samples_leaf=5, max_features='sqrt'):
        self.n_estimators=n_estimators; self.max_depth=max_depth
        self.min_samples_leaf=min_samples_leaf; self.max_features=max_features
    def fit(self, X, y, **kw):
        self.model_ = RandomForestRegressor(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf, max_features=self.max_features,
            random_state=42, n_jobs=-1)
        self.model_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self, 'model_'); return self.model_.predict(X)

class MLPWrapper(BaseEstimator, RegressorMixin):
    _estimator_type = 'regressor'
    def __init__(self, hidden_layer_sizes=(64, 32), alpha=0.01, learning_rate_init=0.001, max_iter=500):
        self.hidden_layer_sizes=hidden_layer_sizes; self.alpha=alpha
        self.learning_rate_init=learning_rate_init; self.max_iter=max_iter
    def fit(self, X, y, **kw):
        self.pipe_ = Pipeline([
            ('scaler', StandardScaler()),
            ('mlp', MLPRegressor(
                hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha,
                learning_rate_init=self.learning_rate_init, max_iter=self.max_iter,
                early_stopping=True, validation_fraction=0.1,
                random_state=42, verbose=False))
        ])
        self.pipe_.fit(X, y); return self
    def predict(self, X):
        check_is_fitted(self, 'pipe_'); return self.pipe_.predict(X)

def make_optuna_tuner(algo, n_trials, max_leaves=MAX_LEAVES):
    tscv = TimeSeriesSplit(n_splits=3)
    def obj_lgb(trial, X, y):
        p = dict(
            n_estimators      = trial.suggest_int('n', 100, 600),
            learning_rate     = trial.suggest_float('lr', 0.005, 0.10, log=True),
            num_leaves        = trial.suggest_int('nl', 8, max_leaves),
            min_child_samples = trial.suggest_int('mcs', 3, 20),
            reg_alpha         = trial.suggest_float('ra', 0.0, 2.0),
            reg_lambda        = trial.suggest_float('rl', 0.5, 3.0))
        m = LGBWrapper(**p); maes = []
        for ti, vi in tscv.split(X):
            if len(X[ti]) < 5: continue
            try:
                mm = copy.deepcopy(m); mm.fit(X[ti], y[ti])
                maes.append(np.mean(np.abs(y[vi] - mm.predict(X[vi]))))
            except: maes.append(1e9)
        return np.mean(maes) if maes else 1e9
    def obj_xgb(trial, X, y):
        p = dict(
            n_estimators   = trial.suggest_int('n', 100, 500),
            learning_rate  = trial.suggest_float('lr', 0.005, 0.10, log=True),
            max_depth      = trial.suggest_int('md', 2, 5),
            min_child_weight = trial.suggest_int('mcw', 3, 15),
            reg_alpha      = trial.suggest_float('ra', 0.0, 2.0),
            reg_lambda     = trial.suggest_float('rl', 0.5, 3.0),
            subsample      = trial.suggest_float('ss', 0.6, 1.0))
        m = XGBWrapper(**p); maes = []
        for ti, vi in tscv.split(X):
            if len(X[ti]) < 5: continue
            try:
                mm = copy.deepcopy(m); mm.fit(X[ti], y[ti])
                maes.append(np.mean(np.abs(y[vi] - mm.predict(X[vi]))))
            except: maes.append(1e9)
        return np.mean(maes) if maes else 1e9
    def obj_cb(trial, X, y):
        p = dict(
            iterations       = trial.suggest_int('n', 100, 500),
            learning_rate    = trial.suggest_float('lr', 0.005, 0.10, log=True),
            depth            = trial.suggest_int('d', 2, 5),
            l2_leaf_reg      = trial.suggest_float('l2', 0.5, 5.0),
            min_data_in_leaf = trial.suggest_int('mdl', 3, 15))
        m = CBWrapper(**p); maes = []
        for ti, vi in tscv.split(X):
            if len(X[ti]) < 5: continue
            try:
                mm = copy.deepcopy(m); mm.fit(X[ti], y[ti])
                maes.append(np.mean(np.abs(y[vi] - mm.predict(X[vi]))))
            except: maes.append(1e9)
        return np.mean(maes) if maes else 1e9
    rename_lgb = {'n':'n_estimators','lr':'learning_rate','nl':'num_leaves',
                  'mcs':'min_child_samples','ra':'reg_alpha','rl':'reg_lambda'}
    rename_xgb = {'n':'n_estimators','lr':'learning_rate','md':'max_depth',
                  'mcw':'min_child_weight','ra':'reg_alpha','rl':'reg_lambda','ss':'subsample'}
    rename_cb  = {'n':'iterations','lr':'learning_rate','d':'depth',
                  'l2':'l2_leaf_reg','mdl':'min_data_in_leaf'}
    def tune(X, y):
        obj_fn = {'lgb': obj_lgb, 'xgb': obj_xgb, 'cb': obj_cb}[algo]
        rename = {'lgb': rename_lgb, 'xgb': rename_xgb, 'cb': rename_cb}[algo]
        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(lambda t: obj_fn(t, X, y), n_trials=n_trials, show_progress_bar=False)
        return {rename.get(k, k): v for k, v in study.best_params.items()}
    return tune

tune_lgb = make_optuna_tuner('lgb', LGB_TRIALS)
tune_xgb = make_optuna_tuner('xgb', XGB_TRIALS)
tune_cb  = make_optuna_tuner('cb',  CB_TRIALS)
print('Wrappers y Optuna definidos')

def tune_rf(X, y):
    tscv = TimeSeriesSplit(n_splits=3)
    def obj(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n', 100, 400),
            max_depth        = trial.suggest_int('md', 4, 12),
            min_samples_leaf = trial.suggest_int('msl', 3, 20),
            max_features     = trial.suggest_categorical('mf', ['sqrt', 'log2', 0.5]))
        m = RFWrapper(**params)
        maes = []
        for ti, vi in tscv.split(X):
            if len(X[ti]) < 5: continue
            try:
                import copy; mm = copy.deepcopy(m); mm.fit(X[ti], y[ti])
                maes.append(np.mean(np.abs(y[vi] - mm.predict(X[vi]))))
            except: maes.append(1e9)
        return np.mean(maes) if maes else 1e9
    st = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    st.optimize(obj, n_trials=RF_TRIALS, show_progress_bar=False)
    rn = {'n':'n_estimators','md':'max_depth','msl':'min_samples_leaf','mf':'max_features'}
    return {rn.get(k,k):v for k,v in st.best_params.items()}

def tune_mlp(X, y):
    tscv = TimeSeriesSplit(n_splits=3)
    def obj(trial):
        h1 = trial.suggest_int('h1', 32, 128)
        h2 = trial.suggest_int('h2', 16, 64)
        params = dict(
            hidden_layer_sizes = (h1, h2),
            alpha              = trial.suggest_float('alpha', 1e-4, 0.1, log=True),
            learning_rate_init = trial.suggest_float('lr', 1e-4, 0.01, log=True),
            max_iter           = 500)
        m = MLPWrapper(**params)
        maes = []
        for ti, vi in tscv.split(X):
            if len(X[ti]) < 5: continue
            try:
                import copy; mm = copy.deepcopy(m); mm.fit(X[ti], y[ti])
                maes.append(np.mean(np.abs(y[vi] - mm.predict(X[vi]))))
            except: maes.append(1e9)
        return np.mean(maes) if maes else 1e9
    st = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    st.optimize(obj, n_trials=MLP_TRIALS, show_progress_bar=False)
    h1 = st.best_params.get('h1', 64); h2 = st.best_params.get('h2', 32)
    return {'hidden_layer_sizes': (h1, h2),
            'alpha': st.best_params.get('alpha', 0.01),
            'learning_rate_init': st.best_params.get('lr', 0.001)}


Wrappers y Optuna definidos


## 3. Carga de Datos y Feature Engineering

In [3]:
df_raw = pd.read_csv('../../../01_Data/processed/1_master_thesis_data.csv')
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw['Match_Key'] = df_raw['Match_Key'].str.lower().str.strip()
df_raw.rename(columns={'Inversión (MMU$)': 'Inversion'}, inplace=True)
df_raw['Production']    = df_raw['Production'].fillna(0).clip(lower=0)
df_raw['Capital_Stock'] = df_raw['Capital_Stock'].fillna(0)
df_raw['Inversion']     = df_raw['Inversion'].fillna(0)
df_raw['Cu_Price']      = df_raw['Cu_Price'].ffill()
df_raw = df_raw[~df_raw['Match_Key'].isin(EXCLUDE_MINES)].reset_index(drop=True)
df_raw = df_raw.sort_values(['Match_Key', 'Date']).reset_index(drop=True)

MINES = sorted(df_raw['Match_Key'].unique())
print(f'Data: {len(df_raw):,} rows | {len(MINES)} minas')

COMPANY_SIZE_MAP = {
    'escondida':2,'chuquicamata':2,'el teniente':2,'andina':2,'radomiro tomic':2,
    'salvador':2,'ministro hales':2,'gabriela mistral':2,'collahuasi':2,
    'los bronces':2,'lomas bayas':2,'cerro colorado':2,'el abra':2,
    'centinela':1,'zaldivar':1,'antucoya':1,'andacollo':1,
    'candelaria':1,'caserones':1,'sierra gorda':1,'los pelambres':1,
    'mantoverde':1,'mantos blancos':1,'el soldado':1,'otros':2,
}
for k in MINES:
    if k not in COMPANY_SIZE_MAP: COMPANY_SIZE_MAP[k] = 0

FIRST_PROD = {}
for mine in MINES:
    sub = df_raw[(df_raw['Match_Key']==mine) & (df_raw['Production']>0)]['Date']
    FIRST_PROD[mine] = sub.min() if len(sub) > 0 else pd.Timestamp('2014-01-01')

PEAK_BY_MONTH = {}
for mine in MINES:
    sub = df_raw[df_raw['Match_Key']==mine].sort_values('Date')
    peak = 0.0; rm = {}
    for _, r in sub.iterrows():
        peak = max(peak, r['Production']); rm[r['Date']] = peak
    PEAK_BY_MONTH[mine] = rm

SIZE_LBL = {0: 'Small', 1: 'Medium', 2: 'Large', 3: 'Colossal'}

def compute_mine_size(df_raw, origin_date):
    end   = origin_date - pd.DateOffset(months=1)
    start = origin_date - pd.DateOffset(months=13)
    avgs  = {}
    for mine in MINES:
        s = df_raw[(df_raw['Match_Key']==mine) &
                   (df_raw['Date'] >= start) & (df_raw['Date'] <= end)]['Production']
        avgs[mine] = s.mean() if (len(s) > 0 and s.sum() > 0) else 0.0
    vals = list(avgs.values())
    q25, q50, q75 = np.percentile(vals, 25), np.percentile(vals, 50), np.percentile(vals, 75)
    return {m: (0 if v<=q25 else (1 if v<=q50 else (2 if v<=q75 else 3))) for m, v in avgs.items()}

def crear_features(df_raw):
    df = df_raw.copy().sort_values(['Match_Key', 'Date']).reset_index(drop=True)
    g  = lambda col: df.groupby('Match_Key')[col]
    df['Prod_Lag1']  = g('Production').shift(1)
    df['Prod_Lag3']  = g('Production').shift(3)
    df['Prod_Lag12'] = g('Production').shift(12)
    df['Prod_Lag24'] = g('Production').shift(24)
    df['Prod_MA12']  = g('Production').transform(
        lambda x: x.shift(1).rolling(12, min_periods=3).mean())
    df['Prod_MA24']  = g('Production').transform(
        lambda x: x.shift(1).rolling(24, min_periods=6).mean())
    df['Prod_MA3']   = g('Production').transform(
        lambda x: x.shift(1).rolling(3, min_periods=2).mean())
    def _trend(s):
        if len(s) < 4: return 0.0
        try: return float(np.polyfit(np.arange(len(s)), s, 1)[0])
        except: return 0.0
    df['Tendencia_12m'] = g('Production').transform(
        lambda x: x.shift(1).rolling(12, min_periods=4).apply(_trend, raw=True))
    df['Tendencia_24m'] = g('Production').transform(
        lambda x: x.shift(1).rolling(24, min_periods=8).apply(_trend, raw=True))
    lag1 = g('Production').shift(1)
    df['Prod_pct_change_m'] = ((df['Production'] - lag1) / (lag1.abs() + 1)).clip(-2, 2)
    lag12 = g('Production').shift(12)
    df['Prod_yoy'] = ((df['Production'] - lag12) / (lag12.abs() + 1)).clip(-2, 2)
    df['Month']     = df['Date'].dt.month
    df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Mine_age']  = df.apply(
        lambda r: max(0, (r['Date'] - FIRST_PROD.get(r['Match_Key'],
                          pd.Timestamp('2014-01-01'))).days / 365.25), axis=1)
    df['Cu_Price_Lag1'] = g('Cu_Price').shift(1)
    df['Cu_Trend_12m']  = g('Cu_Price').transform(
        lambda x: x.shift(1).rolling(12, min_periods=4).apply(_trend, raw=True))
    def get_peak_lag(row):
        pm = PEAK_BY_MONTH.get(row['Match_Key'], {})
        lag_date = row['Date'] - pd.DateOffset(months=1)
        candidates = {d: v for d, v in pm.items() if d <= lag_date}
        return max(candidates.values()) if candidates else 0.0
    df['Prod_vs_peak'] = df.apply(
        lambda r: min(1.0, r['Prod_Lag1'] / (get_peak_lag(r) + 1e-6)), axis=1)
    df['Cu_regime'] = g('Cu_Price').transform(
        lambda x: x.shift(1).rolling(120, min_periods=12).rank(pct=True)).fillna(0.5)
    total_by_month = df.groupby('Date')['Production'].transform('sum')
    df['Mine_share'] = (df['Prod_Lag1'] / (total_by_month.shift(0) + 1)).clip(0, 1)
    df['Company_Size'] = df['Match_Key'].map(COMPANY_SIZE_MAP).fillna(0).astype(int)
    df['Mine_Size']    = 0
    df['Is_Pandemic_Orig'] = ((df['Date'] >= PANDEMIC_START) &
                               (df['Date'] <= PANDEMIC_END)).astype(int)
    # 36-month trend features
    df['Prod_Lag36'] = g('Production').shift(36)
    df['Prod_pct_change_36m'] = ((df['Production'] - df['Prod_Lag36']) / (df['Prod_Lag36'].abs() + 1)).clip(-2, 2)
    def _slope_norm_m(s):
        s = s.dropna()
        if len(s) < 6: return 0.0
        try:
            slope = np.polyfit(np.arange(len(s)), s, 1)[0]
            return float(slope / (s.mean() + 1e-6))
        except: return 0.0
    df['Mine_trend_slope_36m'] = g('Production').transform(
        lambda x: x.shift(1).rolling(36, min_periods=12).apply(_slope_norm_m, raw=False))
    df['Is_RampUp'] = (
        (df['Mine_age'] <= 7) & (df['Prod_pct_change_36m'] > 0.10)
    ).astype(int)
    # NEW: decline state — negative 3-year slope AND below 85% of historical peak
    df['Is_Decline'] = (
        (df['Mine_trend_slope_36m'] < -0.02) & (df['Prod_vs_peak'] < 0.85)
    ).astype(int)
    return df

df_feats = crear_features(df_raw)

# Is_Pandemic_Target added dynamically in ML loop (depends on h).
# For 2026-2032 projections: set both Is_Pandemic_Orig and Is_Pandemic_Target = 0.
E6_BASE  = [
    'Company_Size', 'Mine_Size', 'Prod_Lag1', 'Prod_Lag12', 'Mine_age',
    'Month_sin', 'Month_cos', 'Prod_MA12', 'Tendencia_12m',
    'Prod_pct_change_m', 'Prod_pct_change_36m', 'Mine_trend_slope_36m',
    'Is_RampUp', 'Prod_vs_peak', 'Is_Decline',
    'Cu_regime', 'Mine_share',
    'Is_Pandemic_Orig', 'Is_Pandemic_Target'
]  # 19 features
E6_MULTI = E6_BASE + ['Horizonte_feat']

# OLS-augmented feature set: adds OLS linear prediction as meta-feature
E6_MULTI_OLS = E6_MULTI + ['OLS_Pred']   # 21 features
print(f'Features OK | E6_BASE ({len(E6_BASE)}): {E6_BASE}')

Data: 4,176 rows | 28 minas


Features OK | E6_BASE (19): ['Company_Size', 'Mine_Size', 'Prod_Lag1', 'Prod_Lag12', 'Mine_age', 'Month_sin', 'Month_cos', 'Prod_MA12', 'Tendencia_12m', 'Prod_pct_change_m', 'Prod_pct_change_36m', 'Mine_trend_slope_36m', 'Is_RampUp', 'Prod_vs_peak', 'Is_Decline', 'Cu_regime', 'Mine_share', 'Is_Pandemic_Orig', 'Is_Pandemic_Target']


## 4. Cache ARIMA(1,1,1) por (mina, origen)

In [4]:
def fit_arima_forecast(mine_series, h_max):
    s = mine_series.dropna()
    s = s[s > 0]
    if len(s) < 24:
        return {}
    try:
        model = ARIMA(s.values, order=(1, 1, 1))
        fit   = model.fit()
        fc    = fit.forecast(steps=h_max)
        return {i+1: max(0.0, float(fc[i])) for i in range(h_max)}
    except:
        try:
            model = ARIMA(s.values, order=(0, 1, 0))
            fit   = model.fit()
            fc    = fit.forecast(steps=h_max)
            return {i+1: max(0.0, float(fc[i])) for i in range(h_max)}
        except:
            return {}

print('Pre-calculando forecasts ARIMA(1,1,1)...')
arima_cache = {}
MAX_H_ARIMA = max(HORIZONS)
for origin_date in ROLLING_ORIGINS:
    for mine in MINES:
        series = df_raw[(df_raw['Match_Key'] == mine) &
                        (df_raw['Date'] <= origin_date)].sort_values('Date')['Production']
        arima_cache[(mine, origin_date)] = fit_arima_forecast(series, MAX_H_ARIMA)
    print(f'  {origin_date.strftime("%Y-%m")}', end=' ', flush=True)
print()
print(f'ARIMA cache: {len(arima_cache)} pares (mina, origen)')

Pre-calculando forecasts ARIMA(1,1,1)...


  2017-01 

  2017-07 

  2018-01 

  2018-07 

...

  2020-07 

  2021-01 

  2021-07 

  2022-01 


ARIMA cache: 308 pares (mina, origen)


## 4b. Rolling OLS (expanding window) por Fecha

Para cada mes `t` desde el primer mes con suficiente historia hasta el ultimo rolling origin,
se ajusta un OLS usando **solo datos disponibles hasta ese mes**: `Date <= t AND Target_Date <= t`.
La prediccion OLS para `(mine, t, h)` predice `t+h` meses hacia adelante — genuinamente out-of-sample,
ya que `Target_Date = t+h > t` no fue visto al ajustar el OLS en `t`.

Cache: `ols_pred_cache_m[(date, mine, h)]` → lookup vectorizado en el loop ML.


In [5]:
from sklearn.linear_model import LinearRegression as _OLS

print('Computando rolling OLS mensual (expanding window)...')
ols_pred_cache_m = {}  # (Date, mine, h) -> OLS prediction en LogRatio space

# Rango: primer mes con suficiente historia hasta el ultimo rolling origin
_MIN_OLS_DATE_M = pd.Timestamp('2014-01-01')  # necesita ~36 meses de historia para features
_MAX_OLS_DATE_M = max(ROLLING_ORIGINS)

# Generar lista de meses a iterar (inicio de cada mes)
_months = pd.date_range(_MIN_OLS_DATE_M, _MAX_OLS_DATE_M, freq='MS')
print(f'  Iterando {len(_months)} meses de {_MIN_OLS_DATE_M.strftime("%Y-%m")} a {_MAX_OLS_DATE_M.strftime("%Y-%m")}')

for _td in _months:
    _ms_t = compute_mine_size(df_raw, _td)
    _dfo = df_feats.copy()
    _dfo['Mine_Size'] = _dfo['Match_Key'].map(_ms_t).fillna(0).astype(int)
    _fsm, _flc = [], []
    for _h in HORIZONS:
        _d = _dfo.copy()
        _d['Target']             = _d.groupby('Match_Key')['Production'].shift(-_h)
        _d['Target_Date']        = _d['Date'] + pd.DateOffset(months=_h)
        _d['Horizonte_feat']     = _h
        _d['Is_Pandemic_Target'] = ((_d['Target_Date'] >= PANDEMIC_START) &
                                     (_d['Target_Date'] <= PANDEMIC_END)).astype(int)
        # Strict expanding window: Target_Date <= td (observado antes de predecir en td)
        _sub = _d[(_d['Date'] <= _td) & (_d['Target_Date'] <= _td) & (_d['Prod_Lag1'] > 0)]
        _fsm.append(_sub[_sub['Mine_Size'].isin([0,1])].dropna(
            subset=E6_MULTI + ['Target', 'Production']))
        _flc.append(_sub[_sub['Mine_Size'].isin([2,3])].dropna(
            subset=E6_MULTI + ['Target', 'Production']))

    _trsm = pd.concat(_fsm, ignore_index=True)
    _trlc = pd.concat(_flc, ignore_index=True)
    _ols_sm_t = _ols_lc_t = None
    if len(_trsm) >= 15:
        _ysm = np.clip(np.log((_trsm['Target']+1e-6)/(_trsm['Production']+1e-6)),-1.5,1.5).values
        _ols_sm_t = _OLS().fit(_trsm[E6_MULTI].fillna(0).values, _ysm)
    if len(_trlc) >= 15:
        _ylc = np.clip(np.log((_trlc['Target']+1e-6)/(_trlc['Production']+1e-6)),-1.5,1.5).values
        _ols_lc_t = _OLS().fit(_trlc[E6_MULTI].fillna(0).values, _ylc)

    # Predicciones forward desde td para todos (mine, h)
    for _h in HORIZONS:
        _dh = _dfo.copy()
        _dh['Target']             = _dh.groupby('Match_Key')['Production'].shift(-_h)
        _dh['Target_Date']        = _dh['Date'] + pd.DateOffset(months=_h)
        _dh['Horizonte_feat']     = _h
        _dh['Is_Pandemic_Target'] = ((_dh['Target_Date'] >= PANDEMIC_START) &
                                      (_dh['Target_Date'] <= PANDEMIC_END)).astype(int)
        _rows_t = _dh[(_dh['Date'] == _td) & (_dh['Prod_Lag1'] > 0)]
        for _, _row in _rows_t.iterrows():
            _mine = _row['Match_Key']
            _ms   = int(_row['Mine_Size'])
            _ols_mod = _ols_sm_t if _ms <= 1 else _ols_lc_t
            if _ols_mod is None: continue
            try:
                _x = _row[E6_MULTI].fillna(0).values.reshape(1, -1)
                ols_pred_cache_m[(_td, _mine, _h)] = float(
                    np.clip(_ols_mod.predict(_x), -1.5, 1.5)[0])
            except:
                pass

    if _td.month == 1 or _td == _MAX_OLS_DATE_M:
        _n = sum(1 for k in ols_pred_cache_m if k[0] == _td)
        print(f'  {_td.strftime("%Y-%m")}: SM n={len(_trsm):5d}, LC n={len(_trlc):4d} | preds={_n}')

print(f'Rolling OLS mensual cache: {len(ols_pred_cache_m):,} entradas')

# Vectorized injection helper
def _inject_ols_pred_rolling_m(df, size_filt):
    """Add OLS_Pred via rolling cache. Rows with no cache entry get 0.0."""
    df = df.copy()
    _keys = list(zip(df['Date'], df['Match_Key'], df['Horizonte_feat'].astype(int)))
    df['OLS_Pred'] = [ols_pred_cache_m.get(_k, 0.0) for _k in _keys]
    return df

print('Helper _inject_ols_pred_rolling_m listo.')

Computando rolling OLS mensual (expanding window)...
  Iterando 97 meses de 2014-01 a 2022-01
  2014-01: SM n=    0, LC n=   0 | preds=0


  2015-01: SM n=    0, LC n=   0 | preds=0


  2016-01: SM n=    0, LC n=  22 | preds=182


  2017-01: SM n=    0, LC n= 130 | preds=182


  2018-01: SM n=  473, LC n= 728 | preds=364


  2019-01: SM n= 1422, LC n=1818 | preds=364


  2020-01: SM n= 2707, LC n=3268 | preds=364


  2021-01: SM n= 4325, LC n=5076 | preds=364


  2022-01: SM n= 6281, LC n=7206 | preds=364
Rolling OLS mensual cache: 24,167 entradas
Helper _inject_ols_pred_rolling_m listo.


## 5. Optuna Tuning

In [6]:
print(f'Optuna tuning SmallMed (origin={ROLLING_ORIGINS[-1].strftime("%Y-%m")}, H+{TUNE_H}m)...')
TUNE_ORIGIN = ROLLING_ORIGINS[-1]
ms_tune = compute_mine_size(df_raw, TUNE_ORIGIN)
df_feats_tune = df_feats.copy()
df_feats_tune['Mine_Size'] = df_feats_tune['Match_Key'].map(ms_tune).fillna(0).astype(int)

df_h_tune = df_feats_tune.copy()
df_h_tune['Target']             = df_h_tune.groupby('Match_Key')['Production'].shift(-TUNE_H)
df_h_tune['Target_Date']        = df_h_tune['Date'] + pd.DateOffset(months=TUNE_H)
df_h_tune['Is_Pandemic_Target'] = ((df_h_tune['Target_Date'] >= PANDEMIC_START) &
                                    (df_h_tune['Target_Date'] <= PANDEMIC_END)).astype(int)

tune_df_sm = df_h_tune[
    (df_h_tune['Date'] <= TUNE_ORIGIN) &
    (df_h_tune['Target_Date'] <= TUNE_ORIGIN) &
    (df_h_tune['Mine_Size'].isin([0, 1])) &
    (df_h_tune['Prod_Lag1'] > 0)
].dropna(subset=E6_BASE + ['Target', 'Production'])

optuna_params_sm = {'lgb': {}, 'xgb': {}, 'cb': {}, 'rf': {}, 'mlp': {}}

if len(tune_df_sm) >= 20:
    y_t = np.clip(np.log((tune_df_sm['Target']+1e-6)/(tune_df_sm['Production']+1e-6)).values, -1.5, 1.5)
    X_t = tune_df_sm[E6_BASE].fillna(0).values
    print(f'  Tuning LGB SM (n={len(tune_df_sm)}, {LGB_TRIALS} trials)...')
    optuna_params_sm['lgb'] = tune_lgb(X_t, y_t)
    print(f'  LGB SM OK: {optuna_params_sm["lgb"]}')
    print(f'  Tuning XGB SM ({XGB_TRIALS} trials)...')
    optuna_params_sm['xgb'] = tune_xgb(X_t, y_t)
    print(f'  XGB SM OK: {optuna_params_sm["xgb"]}')
    print(f'  Tuning CB SM ({CB_TRIALS} trials)...')
    optuna_params_sm['cb'] = tune_cb(X_t, y_t)
    print(f'  CB SM OK: {optuna_params_sm["cb"]}')
    print(f'  Tuning RF SM ({RF_TRIALS} trials)...')
    optuna_params_sm['rf'] = tune_rf(X_t, y_t)
    print(f'  RF SM OK: {optuna_params_sm["rf"]}')
    print(f'  Tuning MLP SM ({MLP_TRIALS} trials)...')
    optuna_params_sm['mlp'] = tune_mlp(X_t, y_t)
    print(f'  MLP SM OK: {optuna_params_sm["mlp"]}')
else:
    print(f'  Defaults (n={len(tune_df_sm)})')

# Optuna tuning for LargeColossal [2,3] — separate params
print(f'\nOptuna tuning LargeColossal [2,3] (H+{TUNE_H}m)...')
tune_df_lc = df_h_tune[
    (df_h_tune['Date'] <= TUNE_ORIGIN) &
    (df_h_tune['Target_Date'] <= TUNE_ORIGIN) &
    (df_h_tune['Mine_Size'].isin([2, 3])) &
    (df_h_tune['Prod_Lag1'] > 0)
].dropna(subset=E6_BASE + ['Target', 'Production'])

optuna_params_lc = {'lgb': {}, 'xgb': {}, 'rf': {}, 'mlp': {}}

if len(tune_df_lc) >= 15:
    y_t_lc = np.clip(np.log((tune_df_lc['Target']+1e-6)/(tune_df_lc['Production']+1e-6)).values, -1.5, 1.5)
    X_t_lc = tune_df_lc[E6_BASE].fillna(0).values
    print(f'  Tuning LGB LC (n={len(tune_df_lc)}, {LGB_TRIALS} trials)...')
    optuna_params_lc['lgb'] = tune_lgb(X_t_lc, y_t_lc)
    print(f'  LGB LC OK: {optuna_params_lc["lgb"]}')
    print(f'  Tuning XGB LC ({XGB_TRIALS} trials)...')
    optuna_params_lc['xgb'] = tune_xgb(X_t_lc, y_t_lc)
    print(f'  XGB LC OK: {optuna_params_lc["xgb"]}')
    print(f'  Tuning RF LC ({RF_TRIALS} trials)...')
    optuna_params_lc['rf'] = tune_rf(X_t_lc, y_t_lc)
    print(f'  RF LC OK: {optuna_params_lc["rf"]}')
    print(f'  Tuning MLP LC ({MLP_TRIALS} trials)...')
    optuna_params_lc['mlp'] = tune_mlp(X_t_lc, y_t_lc)
    print(f'  MLP LC OK: {optuna_params_lc["mlp"]}')
else:
    print(f'  Defaults LC (n={len(tune_df_lc)})')

def get_optuna_params(algo_key, size_filt):
    if 2 in size_filt or 3 in size_filt:
        return optuna_params_lc.get(algo_key, {})
    return optuna_params_sm.get(algo_key, {})


Optuna tuning SmallMed (origin=2022-01, H+36m)...
  Tuning LGB SM (n=349, 30 trials)...


  LGB SM OK: {'n_estimators': 107, 'learning_rate': 0.0068876056770360945, 'num_leaves': 18, 'min_child_samples': 11, 'reg_alpha': 1.721498172087237, 'reg_lambda': 2.3653457745156694}
  Tuning XGB SM (15 trials)...


  XGB SM OK: {'n_estimators': 104, 'learning_rate': 0.018635616397116597, 'max_depth': 2, 'min_child_weight': 7, 'reg_alpha': 1.3274653952372222, 'reg_lambda': 1.2909667034276748, 'subsample': 0.7413066576984592}
  Tuning CB SM (15 trials)...


  CB SM OK: {'iterations': 162, 'learning_rate': 0.005950295391592125, 'depth': 5, 'l2_leaf_reg': 3.20501755284444, 'min_data_in_leaf': 12}
  Tuning RF SM (15 trials)...


  RF SM OK: {'n_estimators': 104, 'max_depth': 9, 'min_samples_leaf': 10, 'max_features': 'sqrt'}
  Tuning MLP SM (20 trials)...


  MLP SM OK: {'hidden_layer_sizes': (47, 23), 'alpha': 0.00014936568554617635, 'learning_rate_init': 0.005399484409787433}

Optuna tuning LargeColossal [2,3] (H+36m)...
  Tuning LGB LC (n=410, 30 trials)...


  LGB LC OK: {'n_estimators': 106, 'learning_rate': 0.006895018101786232, 'num_leaves': 17, 'min_child_samples': 4, 'reg_alpha': 1.607286273920493, 'reg_lambda': 1.978794177489314}
  Tuning XGB LC (15 trials)...


  XGB LC OK: {'n_estimators': 105, 'learning_rate': 0.005289780355475602, 'max_depth': 2, 'min_child_weight': 6, 'reg_alpha': 0.48132129005654556, 'reg_lambda': 1.5489494451031058, 'subsample': 0.8847911208853647}
  Tuning RF LC (15 trials)...


  RF LC OK: {'n_estimators': 384, 'max_depth': 8, 'min_samples_leaf': 20, 'max_features': 'log2'}
  Tuning MLP LC (20 trials)...


  MLP LC OK: {'hidden_layer_sizes': (61, 20), 'alpha': 0.01129013355909268, 'learning_rate_init': 0.0007591104805282694}


In [7]:
# Cache mine-size maps once (avoids NameError in Section 6)
_ms_cache_monthly = {od: compute_mine_size(df_raw, od) for od in ROLLING_ORIGINS}
print(f"Mine-size cache ready: {len(_ms_cache_monthly)} origins")

Mine-size cache ready: 11 origins


## 6. Loop ML Rolling-Origin

In [8]:
# ALGO_LIST: SmallMed [0,1] + LargeColossal [2,3]
ALGO_LIST = [
    ('LGB_LogRatio',  'lgb', LGBWrapper, E6_BASE,  [0,1]),
    ('XGB_LogRatio',  'xgb', XGBWrapper, E6_BASE,  [0,1]),
    ('CB_LogRatio',   'cb',  CBWrapper,  E6_BASE,  [0,1]),
    ('LGB_MultiH',    'lgb', LGBWrapper, E6_MULTI, [0,1]),
    ('LGB_LargeCol',  'lgb', LGBWrapper, E6_BASE,  [2,3]),
    ('LGB_MultiH_LC', 'lgb', LGBWrapper, E6_MULTI, [2,3]),
    ('RF_LogRatio',   'rf',  RFWrapper,  E6_BASE,  [0,1]),
    ('RF_MultiH',     'rf',  RFWrapper,  E6_MULTI, [0,1]),
    ('MLP_MultiH',    'mlp', MLPWrapper, E6_MULTI, [0,1]),
    ('XGB_LargeCol',  'xgb', XGBWrapper, E6_BASE,  [2,3]),
    ('RF_LargeCol',   'rf',  RFWrapper,  E6_BASE,  [2,3]),
    ('RF_MultiH_LC',  'rf',  RFWrapper,  E6_MULTI, [2,3]),
    ('MLP_MultiH_LC', 'mlp', MLPWrapper, E6_MULTI, [2,3]),
    # OLS-augmented: ML models receive OLS linear prediction as additional feature
    ('LGB_MultiH_OLS',    'lgb', LGBWrapper, E6_MULTI_OLS, [0,1]),
    ('RF_MultiH_OLS',     'rf',  RFWrapper,  E6_MULTI_OLS, [0,1]),
    ('LGB_MultiH_LC_OLS', 'lgb', LGBWrapper, E6_MULTI_OLS, [2,3]),
    ('RF_MultiH_LC_OLS',  'rf',  RFWrapper,  E6_MULTI_OLS, [2,3]),
]


print(f'Loop ML mensual v7 — {len(ALGO_LIST)} algos × {len(ROLLING_ORIGINS)} orígenes × {len(HORIZONS)} horizontes\n')

ml_records    = []
multi_h_models = {}  # (algo_key, origin_date, seg_key) -> fitted model

for algo_name, algo_key, algo_cls, feat_list, size_filt in ALGO_LIST:
    is_multi = ('Horizonte_feat' in feat_list)
    seg_key  = 'lc' if (2 in size_filt or 3 in size_filt) else 'sm'
    params   = get_optuna_params(algo_key, size_filt)
    # Use Huber objective for SM LGB models (robust to volatile small mines)
    if algo_key == 'lgb' and seg_key == 'sm':
        params = {**params, 'objective': 'huber'}
    print(f'\n=== {algo_name} | multi={is_multi} | size={size_filt} | feats={len(feat_list)} ===')

    for origin_date in ROLLING_ORIGINS:
        ms_map = _ms_cache_monthly[origin_date]   # ← use cache instead of recomputing
        df_feats['Mine_Size'] = df_feats['Match_Key'].map(ms_map).fillna(0).astype(int)
        prod_origin = df_raw[df_raw['Date']==origin_date].groupby('Match_Key')['Production'].mean()

        if is_multi:
            cache_key = (algo_name, origin_date, seg_key)  # use algo_name not algo_key to avoid OLS/non-OLS collision
            if cache_key not in multi_h_models:
                train_frames = []
                for h_tr in HORIZONS:
                    df_ht = df_feats.copy()
                    df_ht['Target']             = df_ht.groupby('Match_Key')['Production'].shift(-h_tr)
                    df_ht['Target_Date']        = df_ht['Date'] + pd.DateOffset(months=h_tr)
                    df_ht['Horizonte_feat']     = h_tr
                    df_ht['Is_Pandemic_Target'] = ((df_ht['Target_Date'] >= PANDEMIC_START) &
                                                    (df_ht['Target_Date'] <= PANDEMIC_END)).astype(int)
                    tr = df_ht[
                        (df_ht['Date'] <= origin_date) &
                        (df_ht['Target_Date'] <= origin_date) &
                        (df_ht['Mine_Size'].isin(size_filt)) &
                        (df_ht['Prod_Lag1'] > 0)
                    ].dropna(subset=[f for f in feat_list if f != 'OLS_Pred'] + ['Target', 'Production'])
                    if len(tr) > 0: train_frames.append(tr)
                if train_frames:
                    train_all = pd.concat(train_frames, ignore_index=True)
                    # OLS_Pred injection for OLS-augmented experiments
                    if 'OLS_Pred' in feat_list:
                        train_all = _inject_ols_pred_rolling_m(train_all, size_filt)
                    if len(train_all) >= 15:
                        y_tr = np.clip(np.log((train_all['Target']+1e-6)/(train_all['Production']+1e-6)).values,-1.5,1.5)
                        X_tr = train_all[feat_list].fillna(0).values
                        m = algo_cls(**params)
                        try: m.fit(X_tr, y_tr); multi_h_models[cache_key] = m
                        except: multi_h_models[cache_key] = None
                    else: multi_h_models[cache_key] = None
                else: multi_h_models[cache_key] = None

            model_multi = multi_h_models.get(cache_key)

            for h in HORIZONS:
                target_date = origin_date + pd.DateOffset(months=h)
                if target_date > pd.Timestamp('2025-12-01'): continue
                df_h = df_feats.copy()
                df_h['Target']             = df_h.groupby('Match_Key')['Production'].shift(-h)
                df_h['Target_Date']        = df_h['Date'] + pd.DateOffset(months=h)
                df_h['Horizonte_feat']     = h
                df_h['Is_Pandemic_Target'] = ((df_h['Target_Date'] >= PANDEMIC_START) &
                                               (df_h['Target_Date'] <= PANDEMIC_END)).astype(int)
                test = df_h[
                    (df_h['Date'] == origin_date) &
                    (df_h['Target_Date'] == target_date) &
                    (df_h['Mine_Size'].isin(size_filt))
                ].dropna(subset=[f for f in feat_list if f != 'OLS_Pred'] + ['Target', 'Production'])
                if 'OLS_Pred' in feat_list and len(test) > 0:
                    test = _inject_ols_pred_rolling_m(test, size_filt)
                    test = test.dropna(subset=['OLS_Pred'])
                if model_multi is None or len(test) == 0: continue
                for _, row in test.iterrows():
                    mine   = row['Match_Key']
                    actual = row['Target']
                    naive  = float(prod_origin.get(mine, np.nan))
                    if pd.isna(naive) or naive == 0: continue
                    ne = abs(actual - naive)
                    origin_prod = row['Production']
                    x = np.array(row[feat_list].fillna(0)).reshape(1, -1)
                    try:
                        raw_pred = float(model_multi.predict(x)[0])
                        pred = max(0, np.exp(raw_pred) * (origin_prod + 1e-6))
                    except: continue
                    me   = abs(actual - pred)
                    mape = abs(actual-pred)/(abs(actual)+1)*100 if actual>0 else np.nan
                    ml_records.append({
                        'Exp': algo_name, 'Origin': origin_date, 'Horizonte': h,
                        'TargetDate': target_date, 'Mine': mine,
                        'Company_Size': int(row['Company_Size']),
                        'Mine_Size': int(row['Mine_Size']),
                        'Actual': actual, 'Pred': pred, 'Naive_Pred': naive,
                        'Model_Error': me, 'Naive_Error': ne,
                        'Beats_Naive': int(me < ne), 'MAPE': mape,
                        'Is_Pandemic_Orig':   int(row['Is_Pandemic_Orig']),
                        'Is_Pandemic_Target': int(row['Is_Pandemic_Target']),
                    })
        else:
            for h in HORIZONS:
                target_date = origin_date + pd.DateOffset(months=h)
                if target_date > pd.Timestamp('2025-12-01'): continue
                df_h = df_feats.copy()
                df_h['Target']             = df_h.groupby('Match_Key')['Production'].shift(-h)
                df_h['Target_Date']        = df_h['Date'] + pd.DateOffset(months=h)
                df_h['Is_Pandemic_Target'] = ((df_h['Target_Date'] >= PANDEMIC_START) &
                                               (df_h['Target_Date'] <= PANDEMIC_END)).astype(int)
                train = df_h[
                    (df_h['Date'] <= origin_date) &
                    (df_h['Target_Date'] <= origin_date) &
                    (df_h['Mine_Size'].isin(size_filt)) &
                    (df_h['Prod_Lag1'] > 0)
                ].dropna(subset=[f for f in feat_list if f != 'OLS_Pred'] + ['Target', 'Production'])
                test = df_h[
                    (df_h['Date'] == origin_date) &
                    (df_h['Target_Date'] == target_date) &
                    (df_h['Mine_Size'].isin(size_filt))
                ].dropna(subset=[f for f in feat_list if f != 'OLS_Pred'] + ['Target', 'Production'])
                if len(train) < 15 or len(test) == 0: continue
                y_tr = np.clip(np.log((train['Target']+1e-6)/(train['Production']+1e-6)).values,-1.5,1.5)
                X_tr = train[feat_list].fillna(0).values
                model = algo_cls(**params)
                try: model.fit(X_tr, y_tr)
                except: model = None
                if model is None: continue
                for _, row in test.iterrows():
                    mine   = row['Match_Key']
                    actual = row['Target']
                    naive  = float(prod_origin.get(mine, np.nan))
                    if pd.isna(naive) or naive == 0: continue
                    ne = abs(actual - naive)
                    origin_prod = row['Production']
                    x = np.array(row[feat_list].fillna(0)).reshape(1, -1)
                    try:
                        raw_pred = float(model.predict(x)[0])
                        pred = max(0, np.exp(raw_pred) * (origin_prod + 1e-6))
                    except: continue
                    me   = abs(actual - pred)
                    mape = abs(actual-pred)/(abs(actual)+1)*100 if actual>0 else np.nan
                    ml_records.append({
                        'Exp': algo_name, 'Origin': origin_date, 'Horizonte': h,
                        'TargetDate': target_date, 'Mine': mine,
                        'Company_Size': int(row['Company_Size']),
                        'Mine_Size': int(row['Mine_Size']),
                        'Actual': actual, 'Pred': pred, 'Naive_Pred': naive,
                        'Model_Error': me, 'Naive_Error': ne,
                        'Beats_Naive': int(me < ne), 'MAPE': mape,
                        'Is_Pandemic_Orig':   int(row['Is_Pandemic_Orig']),
                        'Is_Pandemic_Target': int(row['Is_Pandemic_Target']),
                    })
        print(f'  {algo_name} done: {len([r for r in ml_records if r["Exp"]==algo_name])} records')

df_ml = pd.DataFrame(ml_records)
df_ml.to_csv(f'{EXPORT_DIR}/predicciones_mensuales_baseline_extendido.csv', index=False)
print(f'\nTotal ML records: {len(df_ml):,}')


Loop ML mensual v7 — 17 algos × 11 orígenes × 13 horizontes


=== LGB_LogRatio | multi=False | size=[0, 1] | feats=19 ===


  LGB_LogRatio done: 0 records


  LGB_LogRatio done: 28 records


  LGB_LogRatio done: 84 records


  LGB_LogRatio done: 154 records


...

  RF_MultiH_LC_OLS done: 1365 records


  RF_MultiH_LC_OLS done: 1560 records


  RF_MultiH_LC_OLS done: 1740 records


  RF_MultiH_LC_OLS done: 1905 records


  RF_MultiH_LC_OLS done: 2055 records

Total ML records: 27,172


## 7. Ensemble Post-hoc (LGB_MultiH + ARIMA)

In [9]:
print('Construyendo ensemble ARIMA post-hoc (SmallMed + LargeColossal)...')
ens_records = []

# Best SmallMed base model (auto-selected by Hfoc) — CB_LogRatio excluded
sm_exps = ['LGB_LogRatio', 'XGB_LogRatio', 'LGB_MultiH', 'RF_LogRatio', 'RF_MultiH', 'MLP_MultiH']
best_sm_exp = None; best_sm_hfoc = 0
for exp in sm_exps:
    sub = df_ml[(df_ml['Exp']==exp) & (df_ml['Horizonte'].isin(H_FOCUS))]
    if len(sub) == 0: continue
    hf = 100 * sub['Beats_Naive'].mean()
    if hf > best_sm_hfoc: best_sm_hfoc = hf; best_sm_exp = exp
print(f'  Mejor modelo base SM: {best_sm_exp} (Hfoc={best_sm_hfoc:.1f}%)')

# SmallMed ensembles
if best_sm_exp:
    base_sm = df_ml[df_ml['Exp'] == best_sm_exp].copy()
    for _, row in base_sm.iterrows():
        mine        = row['Mine']
        origin_date = row['Origin']
        h           = row['Horizonte']
        arima_fc    = arima_cache.get((mine, origin_date), {})
        arima_pred  = arima_fc.get(h, np.nan)
        if pd.isna(arima_pred): continue
        for alpha_name, alpha_val in [('Ens_5050', 0.5), ('Ens_7030', 0.7), ('Ens_3070', 0.3)]:
            pred_ens = max(0, alpha_val * row['Pred'] + (1 - alpha_val) * arima_pred)
            me_ens   = abs(row['Actual'] - pred_ens)
            ens_records.append({
                'Exp': alpha_name, 'Origin': origin_date, 'Horizonte': h,
                'TargetDate': row['TargetDate'], 'Mine': mine,
                'Company_Size': row['Company_Size'], 'Mine_Size': row['Mine_Size'],
                'Actual': row['Actual'], 'Pred': pred_ens, 'Naive_Pred': row['Naive_Pred'],
                'Model_Error': me_ens, 'Naive_Error': row['Naive_Error'],
                'Beats_Naive': int(me_ens < row['Naive_Error']),
                'MAPE': abs(row['Actual']-pred_ens)/(abs(row['Actual'])+1)*100,
            })

# LargeColossal ensembles: auto-select best LC base model by Hfoc
lc_candidates_ens = ['LGB_LargeCol', 'XGB_LargeCol', 'RF_LargeCol', 'LGB_MultiH_LC', 'RF_MultiH_LC', 'MLP_MultiH_LC']
best_lc_base = max(
    [e for e in lc_candidates_ens if e in df_ml['Exp'].values],
    key=lambda e: df_ml[(df_ml['Exp']==e) & df_ml['Horizonte'].isin(H_FOCUS)]['Beats_Naive'].mean()
)
lc_hfoc_best = 100 * df_ml[(df_ml['Exp']==best_lc_base) & df_ml['Horizonte'].isin(H_FOCUS)]['Beats_Naive'].mean()
print(f'  Best LC base: {best_lc_base} (Hfoc={lc_hfoc_best:.1f}%)')
lc_base = df_ml[df_ml['Exp'] == best_lc_base].copy()
lc_hfoc = lc_hfoc_best

for _, row in lc_base.iterrows():
    mine        = row['Mine']
    origin_date = row['Origin']
    h           = row['Horizonte']
    arima_fc    = arima_cache.get((mine, origin_date), {})
    arima_pred  = arima_fc.get(h, np.nan)
    if pd.isna(arima_pred): continue
    for alpha_name, alpha_val in [('Ens_LC_5050', 0.5), ('Ens_LC_7030', 0.7), ('Ens_LC_3070', 0.3)]:
        pred_ens = max(0, alpha_val * row['Pred'] + (1 - alpha_val) * arima_pred)
        me_ens   = abs(row['Actual'] - pred_ens)
        ens_records.append({
            'Exp': alpha_name, 'Origin': origin_date, 'Horizonte': h,
            'TargetDate': row['TargetDate'], 'Mine': mine,
            'Company_Size': row['Company_Size'], 'Mine_Size': row['Mine_Size'],
            'Actual': row['Actual'], 'Pred': pred_ens, 'Naive_Pred': row['Naive_Pred'],
            'Model_Error': me_ens, 'Naive_Error': row['Naive_Error'],
            'Beats_Naive': int(me_ens < row['Naive_Error']),
            'MAPE': abs(row['Actual']-pred_ens)/(abs(row['Actual'])+1)*100,
        })

df_ens = pd.DataFrame(ens_records)
df_all = pd.concat([df_ml, df_ens], ignore_index=True)

# Ens_Segmentado: SM → Ens_3070, LC → Ens_LC_3070 (covers ALL mines)
df_sm_ens = df_all[df_all['Exp']=='Ens_3070'].copy()
df_lc_ens = df_all[df_all['Exp']=='Ens_LC_3070'].copy()
seg_records = []
for _, row in df_sm_ens.iterrows():
    d = row.to_dict(); d['Exp'] = 'Ens_Segmentado'; seg_records.append(d)
for _, row in df_lc_ens.iterrows():
    d = row.to_dict(); d['Exp'] = 'Ens_Segmentado'; seg_records.append(d)
df_seg = pd.DataFrame(seg_records)
df_all = pd.concat([df_all, df_seg], ignore_index=True)

# ── Ens_Adaptive: per-mine optimal blend of ML prediction with ARIMA ──
from scipy.optimize import minimize_scalar

adaptive_records = []
for mine in df_all['Mine'].unique():
    is_lc = df_all[(df_all['Mine']==mine) & (df_all['Mine_Size'].isin([2,3]))].shape[0] > 0
    base_exp = 'Ens_LC_3070' if is_lc else 'Ens_3070'
    sub = df_all[(df_all['Exp']==base_exp) & (df_all['Mine']==mine)].copy()
    if len(sub) < 5:
        alpha_m = 0.3
    else:
        def _mae(a, _sub=sub):
            p = a * _sub['Pred'] + (1-a) * _sub['Naive_Pred']
            return np.mean(np.abs(_sub['Actual'] - p))
        res = minimize_scalar(_mae, bounds=(0.0, 1.0), method='bounded')
        alpha_m = round(float(res.x), 3)

    for _, row in sub.iterrows():
        pred_a = max(0, alpha_m * row['Pred'] + (1-alpha_m) * row['Naive_Pred'])
        me_a   = abs(row['Actual'] - pred_a)
        adaptive_records.append({
            **{k: row[k] for k in row.index if k not in ('Exp','Pred','Model_Error','Beats_Naive','MAPE')},
            'Exp': 'Ens_Adaptive',
            'Pred': pred_a,
            'Model_Error': me_a,
            'Beats_Naive': int(me_a < row['Naive_Error']),
            'MAPE': abs(row['Actual']-pred_a)/(abs(row['Actual'])+1)*100 if row['Actual']>0 else np.nan,
            'Alpha_Mine': alpha_m,
        })

df_adaptive = pd.DataFrame(adaptive_records)
df_all = pd.concat([df_all, df_adaptive], ignore_index=True)

print("Mine-specific adaptive alphas (monthly):")
mine_alpha_df = df_adaptive.groupby('Mine')['Alpha_Mine'].first().sort_values()
print(mine_alpha_df.to_string())

df_all.to_csv(f'{EXPORT_DIR}/predicciones_mensuales_baseline_completo.csv', index=False)
print(f'Total registros: {len(df_all):,} → guardado en {EXPORT_DIR}/predicciones_mensuales_baseline_completo.csv')


Construyendo ensemble ARIMA post-hoc (SmallMed + LargeColossal)...


  Mejor modelo base SM: LGB_MultiH (Hfoc=54.1%)
  Best LC base: RF_LargeCol (Hfoc=52.6%)


Mine-specific adaptive alphas (monthly):
Mine
salvador            0.000
los bronces         0.000
sierra gorda        0.230
los pelambres       0.264
cerro colorado      0.397
el teniente         0.401
franke              0.410
candelaria          0.462
andacollo           0.576
chuquicamata        0.839
ojos del salado     0.844
radomiro tomic      1.000
otros               1.000
ministro hales      1.000
mantoverde          1.000
mantos blancos      1.000
escondida           1.000
gabriela mistral    1.000
el soldado          1.000
el abra             1.000
collahuasi          1.000
centinela           1.000
caserones           1.000
atacama kozan       1.000
antucoya            1.000
andina              1.000
lomas bayas         1.000
zaldivar            1.000
Total registros: 42,514 → guardado en outputs_best/predicciones_mensuales_baseline_completo.csv


## 8. Resultados

In [10]:
# Scoreboard Ens_Segmentado
sub_seg = df_all[df_all['Exp']=='Ens_Segmentado'].dropna(subset=['Beats_Naive'])
_hf_sub = sub_seg[sub_seg['Horizonte'].isin(H_FOCUS)]
mine_agg = _hf_sub.groupby('Mine').agg(
    WR=('Beats_Naive','mean'), MAE_M=('Model_Error','mean'),
    MAE_N=('Naive_Error','mean'), MAPE=('MAPE','mean'), MS=('Mine_Size','first')
).reset_index()
mine_agg['MASE']  = (mine_agg['MAE_M'] / mine_agg['MAE_N']).round(3)
mine_agg['Skill'] = ((mine_agg['MAE_N']-mine_agg['MAE_M'])/mine_agg['MAE_N']*100).round(1)

# MdAPE Skill
_mn = _hf_sub.copy()
_mn['Naive_MAPE'] = ((_mn['Actual']-_mn['Naive_Pred']).abs()/(_mn['Actual'].abs()+1)*100)
mine_agg = mine_agg.join(_mn.groupby('Mine')['Naive_MAPE'].median().rename('MdAPE_N'), on='Mine')
mine_agg = mine_agg.join(_hf_sub.groupby('Mine')['MAPE'].median().rename('MdAPE_M'),   on='Mine')
mine_agg['MdAPE_Skill'] = ((mine_agg['MdAPE_N']-mine_agg['MdAPE_M'])/mine_agg['MdAPE_N']*100).round(1)

# Winsorized Skill (recorta top/bottom 5% de errores por mina)
def _wskill_m(grp, trim=0.05):
    em = np.sort(grp['Model_Error'].values)
    en = np.sort(grp['Naive_Error'].values)
    k  = max(1, int(len(em)*trim))
    em_w = em[k:-k] if len(em) > 2*k else em
    en_w = en[k:-k] if len(en) > 2*k else en
    return round((en_w.mean()-em_w.mean())/en_w.mean()*100, 1) if en_w.mean() != 0 else np.nan
mine_agg = mine_agg.join(_hf_sub.groupby('Mine').apply(_wskill_m).rename('WSkill'), on='Mine')

mine_agg['Size_Label'] = mine_agg['MS'].map(SIZE_LBL)
mine_agg = mine_agg.sort_values('MdAPE_Skill', ascending=False)
mine_agg.to_csv(f'{EXPORT_DIR}/scoreboard_monthly_v7.csv', index=False)
print(mine_agg[['Mine','WR','Skill','WSkill','MdAPE_N','MdAPE_M','MdAPE_Skill','MASE','Size_Label']].to_string(index=False))

            Mine       WR  Skill  WSkill    MdAPE_N    MdAPE_M  MdAPE_Skill  MASE Size_Label
          andina 0.666667   33.6    37.0  11.786683   8.772813         25.6 0.664      Large
      el soldado 0.615385   18.4    16.4  15.571070  11.901600         23.6 0.816      Small
   atacama kozan 0.653846   29.9    32.8   6.632060   5.181894         21.9 0.701      Small
      collahuasi 0.777778    9.0    10.1  23.203084  18.278468         21.2 0.910   Colossal
     lomas bayas 0.653846    9.6    10.5  10.321849   8.190317         20.7 0.904     Medium
       centinela 0.666667   23.7    23.0  26.052080  21.059417         19.2 0.763      Large
       escondida 0.777778   15.5    16.9  19.210185  15.674045         18.4 0.845   Colossal
           otros 0.555556   12.8    18.7  21.578043  18.210104         15.6 0.872      Large
        antucoya 0.782609   13.0    13.0   9.747292   8.366310         14.2 0.870     Medium
  ministro hales 0.444444   -4.3    -5.4  30.722019  26.367636        

In [11]:
# ── Color palette (thesis style) ───────────────────────────────────────────
_THESIS_SM = '#4e79a7'
_THESIS_LC = '#9467bd'
_TEAL      = '#0d9488'
_RED       = '#dc2626'
_GRAY      = '#9ca3af'

import matplotlib.pyplot as plt

# ─── Thesis style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.size': 10, 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
})
_BLUE   = '#1d4ed8'
_ORANGE = '#ea580c'
_TEAL   = '#0d9488'
_RED    = '#dc2626'
_GRAY   = '#6b7280'

# ── MdAPE_Skill y Skill por horizonte — Ens_Segmentado mensual ───────────────
_ens_hm  = df_all[df_all['Exp'] == 'Ens_Segmentado'].dropna(subset=['Beats_Naive'])
_all_h_m = sorted(_ens_hm['Horizonte'].unique())
_hm_rows = []
for h in _all_h_m:
    sub_h   = _ens_hm[_ens_hm['Horizonte'] == h]
    wr_h    = sub_h['Beats_Naive'].mean() * 100
    skill_h = (sub_h['Naive_Error'].mean() - sub_h['Model_Error'].mean()) / sub_h['Naive_Error'].mean() * 100
    _naive_ape = (sub_h['Actual'] - sub_h['Naive_Pred']).abs() / (sub_h['Actual'].abs() + 1) * 100
    mdape_n = _naive_ape.median()
    mdape_m = sub_h['MAPE'].median()
    mdape_sk = (mdape_n - mdape_m) / mdape_n * 100
    _em = np.sort(sub_h['Model_Error'].values); _en = np.sort(sub_h['Naive_Error'].values)
    _k  = max(1, int(len(_em) * 0.05))
    _emw = _em[_k:-_k] if len(_em) > 2*_k else _em
    _enw = _en[_k:-_k] if len(_en) > 2*_k else _en
    wsk_h = round((_enw.mean() - _emw.mean()) / _enw.mean() * 100, 1) if _enw.mean() != 0 else np.nan
    _hm_rows.append({'H': h, 'WR%': round(wr_h, 1), 'Skill%': round(skill_h, 1),
                     'WSkill%': wsk_h, 'MdAPE_N': round(mdape_n, 1),
                     'MdAPE_M': round(mdape_m, 1), 'MdAPE_Skill%': round(mdape_sk, 1)})

_df_hbym = pd.DataFrame(_hm_rows)
_df_hbym.to_csv(f'{EXPORT_DIR}/mdape_by_horizon_monthly.csv', index=False)
print(_df_hbym.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────────
hs_m = _df_hbym['H'].values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hs_m, _df_hbym['Skill%'],       'o-',  color=_THESIS_SM,   lw=2,   ms=4, label='Skill% (MAE)')
ax1.plot(hs_m, _df_hbym['MdAPE_Skill%'], 's--', color=_THESIS_LC, lw=1.8, ms=4, label='MdAPE\_Skill%')
ax1.plot(hs_m, _df_hbym['WSkill%'],      '^:',  color=_TEAL,   lw=1.8, ms=4, label='WSkill% (5% trim)')
ax1.fill_between(hs_m, _df_hbym['Skill%'], _df_hbym['MdAPE_Skill%'], alpha=0.08, color=_GRAY)
ax1.axhline(0, color=_GRAY, lw=0.9, ls='--')
for h in H_FOCUS:
    ax1.axvline(h, color=_RED, alpha=0.2, lw=1.2)
ax1.set_xlabel('Horizonte (meses)'); ax1.set_ylabel('Skill (%)')
ax1.set_xticks(hs_m); ax1.tick_params(axis='x', rotation=45)
ax1.set_title('Skill por Horizonte (tres métricas)', fontweight='bold')
ax1.legend()

ax2.plot(hs_m, _df_hbym['WR%'], 'o-', color=_THESIS_SM, lw=2, ms=4, label='WR%')
ax2.axhline(50, color=_RED, lw=1.2, ls='--', label='Umbral 50 %')
for h in H_FOCUS:
    ax2.axvline(h, color=_RED, alpha=0.2, lw=1.2)
ax2.set_xlabel('Horizonte (meses)'); ax2.set_ylabel('Win Rate (%)')
ax2.set_xticks(hs_m); ax2.tick_params(axis='x', rotation=45)
ax2.set_title('Win Rate por Horizonte', fontweight='bold')
ax2.legend()

fig.suptitle('Ens_Segmentado — Desempeño por Horizonte Mensual',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{EXPORT_DIR}/skill_by_horizon_monthly.png', dpi=200)
plt.show()
print(f"Saved → {EXPORT_DIR}/skill_by_horizon_monthly.png")

 H  WR%  Skill%  WSkill%  MdAPE_N  MdAPE_M  MdAPE_Skill%
 1 46.1     2.1      3.4      9.1      8.8           2.9
 3 54.9     4.3      3.8      9.8      9.0           8.4
 6 58.2    12.5     13.8     12.7     10.9          14.6
 9 68.4    18.8     20.0     15.3     11.3          25.7
12 58.1    13.5     15.8     14.8     12.5          16.0
18 64.2     9.7     12.0     16.9     15.1          10.2
24 57.1     3.5      1.9     16.8     15.8           5.6
30 52.0     1.4      1.7     15.7     15.1           4.1
36 55.5     4.8      4.9     18.8     16.7          10.9
42 54.7    -1.1     -4.6     18.8     17.4           7.5
48 46.5    -2.8     -6.4     17.4     17.2           0.8
54 53.2    -1.9     -5.0     18.5     16.9           8.6
60 57.7    -0.7     -2.2     24.2     20.5          15.2


Saved → outputs_best/skill_by_horizon_monthly.png


## 9. Entrenar & Guardar Modelos Finales (origen 2025-12) para Proyecciones

Entrena los modelos definitivos sobre **todos los datos disponibles** (hasta 2025-12) usando los parámetros encontrados por Optuna, y guarda: modelos LGB, forecasts ARIMA, mine sizes y params — para que `generate_projections_monthly.py` cargue exactamente los mismos modelos.

In [12]:
import joblib, json

FINAL_ORIGIN  = pd.Timestamp('2025-12-01')
HORIZONS_PROJ = [1, 3, 6, 9, 12, 18, 24, 30, 36, 42, 48, 54, 60, 72, 84]
FINAL_DIR     = os.path.join(EXPORT_DIR, 'final_models')
os.makedirs(FINAL_DIR, exist_ok=True)

# ── Mine sizes at final origin ─────────────────────────────────────────────
ms_final    = compute_mine_size(df_raw, FINAL_ORIGIN)
df_feats_f  = df_feats.copy()
df_feats_f['Mine_Size'] = df_feats_f['Match_Key'].map(ms_final).fillna(0).astype(int)
size_dist   = {str(v): sum(1 for vv in ms_final.values() if vv == v) for v in [0,1,2,3]}
print(f'Mine sizes at {FINAL_ORIGIN.strftime("%Y-%m")}: {size_dist}')

# ── Build multi-horizon training dataset (todos los datos hasta 2025-12) ───
train_frames_sm, train_frames_lc = [], []
for h_tr in HORIZONS_PROJ:
    df_ht = df_feats_f.copy()
    df_ht['Target']             = df_ht.groupby('Match_Key')['Production'].shift(-h_tr)
    df_ht['Target_Date']        = df_ht['Date'] + pd.DateOffset(months=h_tr)
    df_ht['Horizonte_feat']     = h_tr
    df_ht['Is_Pandemic_Target'] = ((df_ht['Target_Date'] >= PANDEMIC_START) &
                                    (df_ht['Target_Date'] <= PANDEMIC_END)).astype(int)
    tr = df_ht[
        (df_ht['Date'] <= FINAL_ORIGIN) &
        (df_ht['Target_Date'] <= FINAL_ORIGIN) &
        (df_ht['Prod_Lag1'] > 0)
    ].dropna(subset=E6_MULTI + ['Target', 'Production'])
    sm_part = tr[tr['Mine_Size'].isin([0, 1])]
    lc_part = tr[tr['Mine_Size'].isin([2, 3])]
    if len(sm_part) > 0: train_frames_sm.append(sm_part)
    if len(lc_part) > 0: train_frames_lc.append(lc_part)

# ── Train SmallMed [0,1] con params Optuna ─────────────────────────────────
train_sm = pd.concat(train_frames_sm, ignore_index=True)
y_sm = np.clip(np.log((train_sm['Target']+1e-6)/(train_sm['Production']+1e-6)).values, -1.5, 1.5)
X_sm = train_sm[E6_MULTI].fillna(0).values
params_sm_lgb = optuna_params_sm['lgb']
# Use Huber objective for SM (robust to volatile small mines)
final_sm_params = {**params_sm_lgb, 'objective': 'huber'}
final_sm = LGBWrapper(**final_sm_params)
final_sm.fit(X_sm, y_sm)
print(f'\nSmallMed  final: {len(train_sm):,} rows')
print(f'  Optuna params: {final_sm_params}')

# ── Train LargeColossal [2,3] con params Optuna ────────────────────────────
train_lc = pd.concat(train_frames_lc, ignore_index=True)
y_lc = np.clip(np.log((train_lc['Target']+1e-6)/(train_lc['Production']+1e-6)).values, -1.5, 1.5)
X_lc = train_lc[E6_MULTI].fillna(0).values
params_lc_lgb = optuna_params_lc.get('lgb', params_sm_lgb)
final_lc = LGBWrapper(**params_lc_lgb)
final_lc.fit(X_lc, y_lc)
print(f'\nLargeColossal final: {len(train_lc):,} rows')
print(f'  Optuna params: {params_lc_lgb}')

# ── Guardar modelos LGB (solo el booster nativo, sin dependencia de clase) ──
final_sm.model_.booster_.save_model(os.path.join(FINAL_DIR, 'model_sm_final.txt'))
final_lc.model_.booster_.save_model(os.path.join(FINAL_DIR, 'model_lc_final.txt'))
print(f'\nGuardados: model_sm_final.txt, model_lc_final.txt')

# ── Guardar params Optuna + feature list en JSON ───────────────────────────
params_out = {
    'sm':       final_sm_params,
    'lc':       params_lc_lgb,
    'features': E6_MULTI,
    'origin':   FINAL_ORIGIN.strftime('%Y-%m-%d'),
}
with open(os.path.join(FINAL_DIR, 'optuna_params.json'), 'w') as f:
    json.dump(params_out, f, indent=2)
print('Guardado: optuna_params.json')

# ── Guardar mine sizes ─────────────────────────────────────────────────────
pd.DataFrame({'Mine': list(ms_final.keys()),
              'Mine_Size': list(ms_final.values())}
            ).to_csv(os.path.join(FINAL_DIR, 'mine_sizes_final.csv'), index=False)
print('Guardado: mine_sizes_final.csv')

# ── Fit ARIMA(1,1,1) por mina en origen 2025-12 y guardar ─────────────────
print('\nFitting ARIMA(1,1,1) por mina en 2025-12...')
arima_proj = {}
for mine in MINES:
    if mine in EXCLUDE_MINES: continue
    series = df_raw[(df_raw['Match_Key']==mine) &
                    (df_raw['Date'] <= FINAL_ORIGIN)].sort_values('Date')['Production']
    fc = fit_arima_forecast(series, max(HORIZONS_PROJ))
    arima_proj[mine] = fc
    status = f'{len(fc)} steps' if fc else 'FAILED'
    print(f'  {mine:<32} {status}')

arima_rows = [{'Mine': m, 'Horizonte': h, 'ARIMA_Pred': p}
              for m, fc in arima_proj.items() for h, p in fc.items()]
pd.DataFrame(arima_rows).to_csv(
    os.path.join(FINAL_DIR, 'pronosticos_arima_2025.csv'), index=False)
print('Guardado: pronosticos_arima_2025.csv')

print(f'\nTodo guardado en {FINAL_DIR}/')
print('  → Ejecutar generate_projections_monthly.py para crear proyecciones')


Mine sizes at 2025-12: {'0': 7, '1': 7, '2': 7, '3': 7}



SmallMed  final: 15,520 rows
  Optuna params: {'n_estimators': 107, 'learning_rate': 0.0068876056770360945, 'num_leaves': 18, 'min_child_samples': 11, 'reg_alpha': 1.721498172087237, 'reg_lambda': 2.3653457745156694, 'objective': 'huber'}



LargeColossal final: 17,340 rows
  Optuna params: {'n_estimators': 106, 'learning_rate': 0.006895018101786232, 'num_leaves': 17, 'min_child_samples': 4, 'reg_alpha': 1.607286273920493, 'reg_lambda': 1.978794177489314}

Guardados: model_sm_final.txt, model_lc_final.txt
Guardado: optuna_params.json
Guardado: mine_sizes_final.csv

Fitting ARIMA(1,1,1) por mina en 2025-12...
  andacollo                        84 steps
  andina                           84 steps
  antucoya                         84 steps
  atacama kozan                    84 steps
  candelaria                       84 steps
  caserones                        84 steps
  centinela                        84 steps
  cerro colorado                   84 steps
  chuquicamata                     84 steps
  collahuasi                       84 steps
  el abra                          84 steps
  el soldado                       84 steps
  el teniente                      84 steps
  escondida                        84 steps
  franke 

  gabriela mistral                 84 steps
  lomas bayas                      84 steps
  los bronces                      84 steps
  los pelambres                    84 steps
  mantos blancos                   84 steps
  mantoverde                       84 steps
  ministro hales                   84 steps
  ojos del salado                  84 steps
  otros                            84 steps
  radomiro tomic                   84 steps
  salvador                         84 steps
  sierra gorda                     84 steps
  zaldivar                         84 steps
Guardado: pronosticos_arima_2025.csv

Todo guardado en outputs_best/final_models/
  → Ejecutar generate_projections_monthly.py para crear proyecciones


## Sección 9b — Optuna Full-Data Tuning (para proyecciones)

Re-entrena Optuna usando el **dataset completo** `(X_sm, y_sm)` y `(X_lc, y_lc)` de Sección 9
(todos los datos hasta Dic 2025, todos los horizontes H+1→H+60 concatenados).
Actualiza `optuna_params_sm['lgb']` y `optuna_params_lc['lgb']` para que Sección 10 use
los mejores parámetros sobre el máximo de datos disponible.
Guarda `optuna_params_monthly.json` para reproducibilidad (igual que el modelo anual).

In [13]:
import json as _json_m

print("Optuna full-data tuning para proyecciones mensuales...")
print(f"  Datos: SmallMed n={len(X_sm):,} | LargeColossal n={len(X_lc):,}")

# ── SmallMed: usa X_sm, y_sm de Sección 9 (E6_MULTI, todos los horizontes) ──
if len(X_sm) >= 20:
    print(f"\n  Tuning LGB SM ({LGB_TRIALS} trials)...")
    optuna_params_sm['lgb'] = tune_lgb(X_sm, y_sm)
    print(f"  → {optuna_params_sm['lgb']}")
else:
    print(f"  SM: n={len(X_sm)} < 20, saltando (conserva params del loop de validación)")

# ── LargeColossal: usa X_lc, y_lc de Sección 9 ───────────────────────────────
if len(X_lc) >= 20:
    print(f"\n  Tuning LGB LC ({LGB_TRIALS} trials)...")
    optuna_params_lc['lgb'] = tune_lgb(X_lc, y_lc)
    print(f"  → {optuna_params_lc['lgb']}")
else:
    print(f"  LC: n={len(X_lc)} < 20, saltando (conserva params del loop de validación)")

# ── Guardar JSON ──────────────────────────────────────────────────────────────
_monthly_params_out = {
    'sm':       optuna_params_sm['lgb'],
    'lc':       optuna_params_lc['lgb'],
    'features': E6_MULTI,
    'origin':   str(FINAL_ORIGIN.date()),
}
_out_path = os.path.join(EXPORT_DIR, 'optuna_params_monthly.json')
with open(_out_path, 'w') as _f:
    _json_m.dump(_monthly_params_out, _f, indent=2)
print(f"\nGuardado: {_out_path}")

Optuna full-data tuning para proyecciones mensuales...
  Datos: SmallMed n=15,520 | LargeColossal n=17,340

  Tuning LGB SM (30 trials)...


  → {'n_estimators': 242, 'learning_rate': 0.09013710971500914, 'num_leaves': 18, 'min_child_samples': 3, 'reg_alpha': 0.37205247339245834, 'reg_lambda': 1.4068723065690807}

  Tuning LGB LC (30 trials)...


  → {'n_estimators': 433, 'learning_rate': 0.06245042298040794, 'num_leaves': 18, 'min_child_samples': 11, 'reg_alpha': 1.056611039986717, 'reg_lambda': 1.0406329614937788}

Guardado: outputs_best/optuna_params_monthly.json


## Sección 10 — Proyecciones Mensuales 2026-2032

Genera proyecciones mensuales de producción H+1 a H+84 con:
- **Lags recursivos**: `Prod_Lag1`, `Prod_Lag12`, `Prod_MA12`, `Tendencia_12m` se actualizan con las predicciones anteriores (en vez de estar congelados en Dic 2025)
- **Bandas de confianza reales**: LightGBM cuantílico q10/q90 aplicado también de forma recursiva
- **Escenarios de precio Cu**: bear (Cu_regime=0.2), base (0.5), bull (0.8)
- Salida principal: `proyecciones_mensuales_2026_2032.csv` (escenario base, compatible con el dashboard)
- Salida adicional: `proyecciones_escenarios_mensuales_2026_2032.csv` (los 3 escenarios)


In [14]:
import math

MAX_H_PROJ     = max(HORIZONS_PROJ)    # 84
TARGET_H_SET   = set(HORIZONS_PROJ)
CU_SCENARIOS_M = {"bear": 0.2, "base": 0.5, "bull": 0.8}

# ── Train q10/q90 quantile models (reuse X_sm, X_lc from Sección 9) ──────────
def _mon_trio(X, y, params, name, use_huber=False):
    base = {**params, 'random_state': 42, 'verbose': -1}
    if use_huber:
        base['objective'] = 'huber'
    m    = lgb.LGBMRegressor(**base).fit(X, y)
    m10  = lgb.LGBMRegressor(**{**{k: v for k, v in base.items() if k != 'objective'},
                                 'objective': 'quantile', 'alpha': 0.1}).fit(X, y)
    m90  = lgb.LGBMRegressor(**{**{k: v for k, v in base.items() if k != 'objective'},
                                 'objective': 'quantile', 'alpha': 0.9}).fit(X, y)
    print(f"  {name}: mean + q10 + q90 trained")
    return m, m10, m90

print("Training monthly quantile projection models...")
_p_sm = optuna_params_sm['lgb']
_p_lc = optuna_params_lc.get('lgb', _p_sm)
q_sm_m,  q_sm10_m,  q_sm90_m  = _mon_trio(X_sm, y_sm, _p_sm, "SmallMed [0,1]", use_huber=True)
q_lc_m,  q_lc10_m,  q_lc90_m  = _mon_trio(X_lc, y_lc, _p_lc, "LargeColossal [2,3]")

# ── Recursive projection per mine ─────────────────────────────────────────────
def _trend_r(s):
    if len(s) < 4: return 0.0
    try: return float(np.polyfit(np.arange(len(s)), np.array(s, dtype=float), 1)[0])
    except: return 0.0

def _slope_norm_r(s):
    s = [v for v in s if v is not None and not (isinstance(v, float) and math.isnan(v))]
    if len(s) < 6: return 0.0
    try:
        arr = np.array(s, dtype=float)
        slope = np.polyfit(np.arange(len(arr)), arr, 1)[0]
        return float(slope / (arr.mean() + 1e-6))
    except: return 0.0

def _project_recursive(mine, op, origin_row, ms, trio, cu_val, mine_age_0):
    """Recursive multi-step forecast: each prediction updates lag features."""
    m, m10, m90 = trio
    _hist_raw = df_raw[(df_raw['Match_Key'] == mine) &
                       (df_raw['Date'] <= FINAL_ORIGIN)].sort_values('Date')['Production'].values
    combined = list(_hist_raw[-36:]) if len(_hist_raw) >= 36 else list(_hist_raw)
    if not combined: combined = [op]

    _co = float(origin_row.get('Company_Size', 0))
    _sh = float(origin_row.get('Mine_share', 0))

    def _run(model):
        hist = list(combined)
        peak = max(hist) if hist else op   # running historical max
        out  = {}
        for h in range(1, MAX_H_PROJ + 1):
            td    = FINAL_ORIGIN + pd.DateOffset(months=h)
            lag1  = hist[-1]
            lag12 = hist[-12] if len(hist) >= 12 else hist[0]
            lag36 = hist[-36] if len(hist) >= 36 else hist[0]
            last12 = hist[-12:] if len(hist) >= 12 else hist
            last36 = hist[-36:] if len(hist) >= 36 else hist
            ma12  = float(np.mean(last12))
            tend  = _trend_r(last12)
            pct   = float(np.clip(
                (lag1 - hist[-2]) / (abs(hist[-2]) + 1) if len(hist) >= 2 else 0.0,
                -2, 2))
            pct36 = float(np.clip((lag1 - lag36) / (abs(lag36) + 1), -2, 2))
            slope36 = _slope_norm_r(last36)
            _age   = mine_age_0 + h / 12.0
            is_ru  = int(_age <= 7 and pct36 > 0.10)
            pvp    = min(1.0, lag1 / (peak + 1e-6))          # capacity ceiling
            is_dec = int(slope36 < -0.02 and pvp < 0.85)     # decline state
            x_dict = {
                'Company_Size':         _co,
                'Mine_Size':            float(ms),
                'Prod_Lag1':            float(lag1),
                'Prod_Lag12':           float(lag12),
                'Mine_age':             _age,
                'Prod_MA12':            ma12,
                'Tendencia_12m':        tend,
                'Prod_pct_change_m':    pct,
                'Prod_pct_change_36m':  pct36,
                'Mine_trend_slope_36m': slope36,
                'Is_RampUp':            float(is_ru),
                'Prod_vs_peak':         pvp,
                'Is_Decline':           float(is_dec),
                'Cu_regime':            cu_val,
                'Mine_share':           _sh,
                'Is_Pandemic_Orig':     0.0,
                'Is_Pandemic_Target':   0.0,
                'Month_sin':            float(np.sin(2 * np.pi * td.month / 12)),
                'Month_cos':            float(np.cos(2 * np.pi * td.month / 12)),
                'Horizonte_feat':       float(h),
            }
            x_feat = np.array([x_dict[f] for f in E6_MULTI], dtype=float).reshape(1, -1)
            log_ratio = float(model.predict(x_feat)[0])
            pred = max(0.0, math.exp(log_ratio) * (op + 1e-6))
            hist.append(pred)
            peak = max(peak, pred)   # update running max
            if h in TARGET_H_SET:
                out[h] = pred
        return out

    return _run(m), _run(m10), _run(m90)

# ── Main projection loop ───────────────────────────────────────────────────────
print(f"\nGenerating recursive projections H+1→H+{MAX_H_PROJ} from {FINAL_ORIGIN.strftime('%Y-%m')}...")

origin_rows_m = df_feats[df_feats['Date'] == FINAL_ORIGIN].copy()
origin_rows_m['Mine_Size']    = origin_rows_m['Match_Key'].map(ms_final).fillna(1).astype(int)
origin_rows_m['Company_Size'] = origin_rows_m['Match_Key'].map(COMPANY_SIZE_MAP).fillna(0).astype(int)

all_proj_m = []
for _scenario, _cu in CU_SCENARIOS_M.items():
    print(f"  Scenario: {_scenario} (Cu_regime={_cu})", end='  ')
    for _, _row in origin_rows_m.iterrows():
        _mine = _row['Match_Key']
        if _mine in EXCLUDE_MINES: continue
        _op = float(_row['Production'])
        if _op <= 0 or pd.isna(_row.get('Prod_Lag1')): continue
        _ms   = int(_row['Mine_Size'])
        _trio = (q_sm_m, q_sm10_m, q_sm90_m) if _ms <= 1 else (q_lc_m, q_lc10_m, q_lc90_m)
        _ma0  = float(_row.get('Mine_age', 0))
        _r_med, _r_lo, _r_hi = _project_recursive(_mine, _op, _row, _ms, _trio, _cu, _ma0)
        for h in HORIZONS_PROJ:
            if h not in _r_med: continue
            td = FINAL_ORIGIN + pd.DateOffset(months=h)
            all_proj_m.append({
                'Mine':         _mine,
                'ForecastDate': td.strftime('%Y-%m-%d'),
                'Horizonte':    h,
                'Scenario':     _scenario,
                'Pred':         round(_r_med[h],          3),
                'Naive_Pred':   round(_op,                3),
                'Lower':        round(max(0.0, _r_lo[h]), 3),
                'Upper':        round(_r_hi[h],           3),
                'Origin_Prod':  round(_op,                3),
                'Mine_Size':    _ms,
                'Size_Label':   SIZE_LBL[_ms],
                'Company_Size': int(_row['Company_Size']),
                'Segment':      'SmallMed' if _ms <= 1 else 'LargeColossal',
                'Cu_Regime':    _cu,
            })
    print("done")

df_proj_m_all = pd.DataFrame(all_proj_m)

# Base scenario → dashboard-compatible
OUT_MON  = os.path.join(EXPORT_DIR, 'proyecciones_mensuales_2026_2032.csv')
_base_m  = df_proj_m_all[df_proj_m_all['Scenario'] == 'base'].drop(columns=['Scenario','Cu_Regime'])
_base_m.to_csv(OUT_MON, index=False)

# All scenarios
df_proj_m_all.to_csv(os.path.join(EXPORT_DIR, 'proyecciones_escenarios_mensuales_2026_2032.csv'), index=False)

print(f"\nProjections saved → {EXPORT_DIR}/")
print(f"  proyecciones_mensuales_2026_2032.csv           : {len(_base_m)} rows | {_base_m['Mine'].nunique()} mines | recursive + q10/q90")
print(f"  proyecciones_escenarios_mensuales_2026_2032.csv : {len(df_proj_m_all)} rows | 3 scenarios")

Training monthly quantile projection models...


  SmallMed [0,1]: mean + q10 + q90 trained


  LargeColossal [2,3]: mean + q10 + q90 trained

Generating recursive projections H+1→H+84 from 2025-12...
  Scenario: bear (Cu_regime=0.2)  

done
  Scenario: base (Cu_regime=0.5)  

done
  Scenario: bull (Cu_regime=0.8)  

done

Projections saved → outputs_best/
  proyecciones_mensuales_2026_2032.csv           : 420 rows | 27 mines | recursive + q10/q90
  proyecciones_escenarios_mensuales_2026_2032.csv : 1260 rows | 3 scenarios


## Sección 10b — Test de Diebold-Mariano por mina

Evalúa si la diferencia de precisión entre **Ens_Segmentado** y la predicción naïve es estadísticamente significativa usando el test DM con función de pérdida de error cuadrático (H0: igual precisión predictiva). Horizontes de foco: H+36, H+48, H+60.

- **DM < 0 + p < 0.10** → modelo significativamente mejor que naïve
- **DM > 0 + p < 0.10** → naïve significativamente mejor
- **TIE** → diferencia no significativa


In [15]:
from scipy import stats

def _dm_test_m(e_model, e_naive):
    """Diebold-Mariano test (H0: equal predictive accuracy, squared-error loss).
    Negative DM stat → model better than naive."""
    d = np.array(e_model)**2 - np.array(e_naive)**2
    n = len(d)
    if n < 4: return np.nan, np.nan
    d_bar = np.mean(d)
    var_d = np.var(d, ddof=1) / n
    if var_d <= 0: return 0.0, 1.0
    dm  = d_bar / np.sqrt(var_d)
    p   = 2 * float(stats.norm.sf(abs(dm)))
    return round(float(dm), 3), round(p, 4)

df_dm_input_m = df_all[(df_all['Exp'] == 'Ens_Segmentado') &
                        (df_all['Horizonte'].isin(H_FOCUS))].dropna(subset=['Model_Error','Naive_Error'])

dm_rows_mon = []
for mine, grp in df_dm_input_m.groupby('Mine'):
    dm_stat, p_val = _dm_test_m(grp['Model_Error'].values, grp['Naive_Error'].values)
    dm_rows_mon.append({
        'Mine':    mine,
        'n':       len(grp),
        'WR_%':    round(grp['Beats_Naive'].mean() * 100, 1),
        'DM_stat': dm_stat,
        'p_value': p_val,
        'sig':     '***' if (isinstance(p_val, float) and p_val < 0.01) else
                   ('**'  if (isinstance(p_val, float) and p_val < 0.05) else
                   ('*'   if (isinstance(p_val, float) and p_val < 0.10) else '')),
        'verdict': 'MODEL★' if (isinstance(dm_stat, float) and dm_stat < 0 and isinstance(p_val, float) and p_val < 0.10) else
                   ('NAIVE★' if (isinstance(dm_stat, float) and dm_stat > 0 and isinstance(p_val, float) and p_val < 0.10) else 'TIE'),
    })

df_dm_mon = pd.DataFrame(dm_rows_mon).sort_values('DM_stat')

print('Diebold-Mariano Test — Ens_Segmentado vs Naive | H+36/48/60m | Squared-error loss')
print('H0: equal predictive accuracy  |  Negative DM → model better than naive')
print(f'\n  {"Mine":<38} {"n":>5} {"WR%":>6} {"DM":>8} {"p":>8} {"sig":>4} {"verdict":>8}')
print('  ' + '-'*82)
for _, r in df_dm_mon.iterrows():
    print(f'  {r["Mine"]:<38} {r["n"]:>5} {r["WR_%"]:>5.1f}% {r["DM_stat"]:>8.3f} '
          f'{r["p_value"]:>8.4f} {r["sig"]:>4} {r["verdict"]:>8}')

n_sig = int((df_dm_mon['p_value'] < 0.10).sum())
n_mod = int((df_dm_mon['verdict'].str.startswith('MODEL')).sum())
print(f'\n  Significant (p<0.10): {n_sig}/{len(df_dm_mon)} | Model significantly better: {n_mod}/{len(df_dm_mon)}')

# Append DM stats to scoreboard CSV
_sb_mon = pd.read_csv(os.path.join(EXPORT_DIR, 'scoreboard_monthly_v7.csv'))
_sb_mon['Mine'] = _sb_mon['Mine'].str.lower().str.strip()
df_dm_mon['Mine'] = df_dm_mon['Mine'].str.lower().str.strip()
_sb_mon = _sb_mon.merge(df_dm_mon[['Mine','DM_stat','p_value','sig','verdict']], on='Mine', how='left')
_sb_mon.to_csv(os.path.join(EXPORT_DIR, 'scoreboard_monthly_v7.csv'), index=False)
print(f'\nScoreboard actualizado con stats DM → scoreboard_monthly_v7.csv')


Diebold-Mariano Test — Ens_Segmentado vs Naive | H+36/48/60m | Squared-error loss
H0: equal predictive accuracy  |  Negative DM → model better than naive

  Mine                                       n    WR%       DM        p  sig  verdict
  ----------------------------------------------------------------------------------
  centinela                                 18  66.7%   -2.877   0.0040  ***   MODEL★
  lomas bayas                               26  65.4%   -2.857   0.0043  ***   MODEL★
  antucoya                                  23  78.3%   -2.828   0.0047  ***   MODEL★
  atacama kozan                             26  65.4%   -2.642   0.0082  ***   MODEL★
  escondida                                  9  77.8%   -2.175   0.0296   **   MODEL★
  el soldado                                26  61.5%   -2.162   0.0306   **   MODEL★
  andina                                     9  66.7%   -2.052   0.0402   **   MODEL★
  mantos blancos                            26  65.4%   -2.043   0.0411 

In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# Sección 10b-bis — Corrección de Sesgo Post-hoc por Mina (Mensual)
# Calcula el sesgo medio (Pred − Actual) de Ens_Segmentado en validación
# H+36..H+60 y lo resta de las proyecciones mensuales 2026-2032.
# ═══════════════════════════════════════════════════════════════════════════════
seg_val_m = df_all[(df_all['Exp'] == 'Ens_Segmentado') &
                   (df_all['Actual'] > 0) &
                   (df_all['Horizonte'] >= 36)].copy()
seg_val_m['residual']     = seg_val_m['Pred'] - seg_val_m['Actual']
seg_val_m['rel_residual'] = seg_val_m['residual'] / seg_val_m['Actual']

bias_stats_m = (
    seg_val_m.groupby('Mine')
             .agg(
                 Bias_kt  =('residual',     'mean'),
                 Bias_pct =('rel_residual', lambda x: x.mean() * 100),
                 WR       =('Beats_Naive',  'mean'),
                 N        =('residual',     'count'),
             )
             .reset_index()
             .sort_values('Bias_kt')
)
print("Per-mine mean bias (Pred − Actual, kt/month) H+36..H+60 — Ens_Segmentado:")
print(bias_stats_m[['Mine','Bias_kt','Bias_pct','WR','N']].to_string(index=False))

mine_bias_m = bias_stats_m.set_index('Mine')['Bias_kt'].to_dict()

for _path in [os.path.join(EXPORT_DIR, 'proyecciones_mensuales_2026_2032.csv'),
              os.path.join(EXPORT_DIR, 'proyecciones_escenarios_mensuales_2026_2032.csv')]:
    _df   = pd.read_csv(_path)
    _corr = _df['Mine'].map(mine_bias_m).fillna(0)
    _df['Pred']  = (_df['Pred']  - _corr).clip(lower=0)
    _df['Lower'] = (_df['Lower'] - _corr).clip(lower=0)
    _df['Upper'] = (_df['Upper'] - _corr).clip(lower=0)
    _df.to_csv(_path, index=False)
    print(f"Bias-corrected: {_path.split('/')[-1]}")

bias_stats_m.to_csv(os.path.join(EXPORT_DIR, 'bias_correction_monthly.csv'), index=False)
print(f"\nTop overestimation: {bias_stats_m[bias_stats_m['Bias_kt']>0][['Mine','Bias_kt']].tail(5).to_string(index=False)}")
print(f"Top underestimation: {bias_stats_m[bias_stats_m['Bias_kt']<0][['Mine','Bias_kt']].head(5).to_string(index=False)}")

Per-mine mean bias (Pred − Actual, kt/month) H+36..H+60 — Ens_Segmentado:
            Mine   Bias_kt   Bias_pct       WR  N
       escondida -6.695510  -4.648519 0.800000 15
    sierra gorda -1.821489 -11.186911 0.457143 35
      candelaria -1.748642 -15.413920 0.363636 22
  radomiro tomic -1.581894  -4.805192 0.733333 15
         el abra -1.314413 -13.249504 0.613636 44
      mantoverde -1.244337 -16.281639 0.613636 44
        antucoya -0.670713  -9.750113 0.743590 39
  mantos blancos -0.281625  -3.770740 0.659091 44
   atacama kozan -0.047215  -2.168159 0.750000 44
      el soldado  0.038518   5.419778 0.636364 44
     lomas bayas  0.264897   6.087202 0.613636 44
       caserones  0.302291  10.035882 0.666667 15
          franke  0.440216  48.351538 0.636364 44
gabriela mistral  0.465312   8.844263 0.458333 24
       centinela  0.858412  15.283075 0.733333 30
 ojos del salado  0.866371  71.901780 0.454545 44
          andina  1.117274   8.518854 0.666667 15
        zaldivar  1.514395

In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# Sección 10c — ¿Por qué algunas minas son más predecibles? Análisis diagnóstico
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.stats import spearmanr

sb_m = pd.read_csv(os.path.join(EXPORT_DIR, 'scoreboard_monthly_v7.csv'))
sb_m['Mine'] = sb_m['Mine'].str.lower().str.strip()

ms_last  = compute_mine_size(df_raw, pd.Timestamp('2022-01-01'))
last_orig = df_feats[df_feats['Date'] == pd.Timestamp('2022-01-01')].copy()
last_orig['Mine']      = last_orig['Match_Key'].str.lower().str.strip()
last_orig['Mine_Size'] = last_orig['Mine'].map(ms_last).fillna(1).astype(int)
feat_m = last_orig[['Mine','Mine_Size','Mine_age','Prod_vs_peak','Is_Decline',
                     'Is_RampUp','Mine_trend_slope_36m','Mine_share']].copy()

analysis_m = sb_m.merge(feat_m, on='Mine', how='left')
analysis_m['Tier'] = pd.cut(analysis_m['WR'],
    bins=[0, 0.40, 0.55, 0.70, 1.01],
    labels=['Pobre (<40%)', 'Regular (40-55%)', 'Bueno (55-70%)', 'Excelente (>70%)'])

print("=" * 72)
print("PREDICTIBILIDAD MENSUAL — DIAGNÓSTICO DE FACTORES")
print("=" * 72)
for tier, grp in analysis_m.groupby('Tier', observed=True):
    print(f"\n── {tier}  (n={len(grp)}) ──")
    print(f"  WR: {grp['WR'].mean()*100:.1f}%  Skill: {grp['Skill'].mean():.1f}%")
    ms_mode = int(grp['Mine_Size'].mode().iloc[0]) if len(grp) > 0 else 0
    print(f"  Mine_Size modal: {ms_mode} ({SIZE_LBL.get(ms_mode,'?')})")
    print(f"  Prod_vs_peak: {grp['Prod_vs_peak'].median():.2f}  Is_Decline: {grp['Is_Decline'].mean()*100:.0f}%  Is_RampUp: {grp['Is_RampUp'].mean()*100:.0f}%  Mine_age: {grp['Mine_age'].median():.1f}y")
    print(f"  Minas: {', '.join(sorted(grp['Mine'].tolist()))}")

print("\n── Spearman ρ: WR vs características ──────────────────────────────────")
for col in ['Mine_Size','Mine_age','Prod_vs_peak','Is_Decline','Is_RampUp',
            'Mine_trend_slope_36m','Mine_share']:
    sub = analysis_m[['WR', col]].dropna()
    if len(sub) < 5: continue
    r, p = spearmanr(sub['WR'], sub[col])
    sig  = '***' if p<0.01 else ('**' if p<0.05 else ('*' if p<0.10 else '   '))
    print(f"  {col:<28}: ρ={r:+.3f}  p={p:.3f} {sig}")

print("\n── Conclusión ──────────────────────────────────────────────────────────")
print("  · Minas con Is_Decline=1 tienen trayectoria estable → predecibles.")
print("  · Colossal (Mine_Size=3): expansiones discretas como QB2 → WR bajo.")
print("  · Is_RampUp=1: minas jóvenes con alta incertidumbre de rampa.")
print("  · MAPE>100% (cerro_colorado, salvador): producción cercana a cero,")
print("    los errores relativos se inflan — usar MASE como métrica alternativa.")

PREDICTIBILIDAD MENSUAL — DIAGNÓSTICO DE FACTORES

── Pobre (<40%)  (n=7) ──
  WR: 24.6%  Skill: -15.8%
  Mine_Size modal: 1 (Medium)
  Prod_vs_peak: 0.72  Is_Decline: 0%  Is_RampUp: 0%  Mine_age: 8.0y
  Minas: candelaria, cerro colorado, el teniente, gabriela mistral, los bronces, los pelambres, salvador

── Regular (40-55%)  (n=5) ──
  WR: 47.5%  Skill: -3.0%
  Mine_Size modal: 0 (Small)
  Prod_vs_peak: 0.79  Is_Decline: 0%  Is_RampUp: 0%  Mine_age: 8.0y
  Minas: mantoverde, ministro hales, ojos del salado, sierra gorda, zaldivar

── Bueno (55-70%)  (n=14) ──
  WR: 63.1%  Skill: 14.1%
  Mine_Size modal: 0 (Small)
  Prod_vs_peak: 0.64  Is_Decline: 7%  Is_RampUp: 0%  Mine_age: 8.0y
  Minas: andacollo, andina, atacama kozan, caserones, centinela, centinela, chuquicamata, el abra, el soldado, franke, lomas bayas, mantos blancos, otros, radomiro tomic

── Excelente (>70%)  (n=3) ──
  WR: 77.9%  Skill: 12.5%
  Mine_Size modal: 3 (Colossal)
  Prod_vs_peak: 0.83  Is_Decline: 0%  Is_RampUp: 0

## Sección 11 — Interpretabilidad: SHAP Analysis

Usa SHAP (SHapley Additive exPlanations) con `TreeExplainer` sobre los modelos LightGBM finales de ambos segmentos (`q_sm_m` para SmallMed, `q_lc_m` para LargeColossal).

**Tres visualizaciones:**
1. **Importancia global (bar)** — magnitud promedio de impacto en la predicción
2. **Beeswarm** — dirección del efecto: rojo = valor alto de feature, azul = valor bajo
3. **Importancia por bucket de horizonte** — cómo cambia la relevancia de cada feature a corto (H+1–12), mediano (H+13–36) y largo plazo (H+37–60)

El eje X de SHAP representa el cambio en el log-ratio predicho (≈ cambio porcentual en producción respecto al origen).


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 11 — SHAP Analysis: Interpretabilidad del modelo LightGBM Mensual
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, warnings
warnings.filterwarnings('ignore')
try:
    import shap
except ImportError:
    subprocess.run(['pip', 'install', '--quiet', 'shap'], check=True)
    import shap

import matplotlib.pyplot as plt
import numpy as np

# ─── Thesis style ─────────────────────────────────────────────────────────────
_THESIS = {
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'axes.labelcolor': '#111827',
    'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.family': 'DejaVu Sans',
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
}
plt.rcParams.update(_THESIS)

_THESIS_SM = '#4e79a7'
_THESIS_LC = '#9467bd'
_BUCKET_COLORS = ['#1d4ed8', '#0d9488', '#ea580c', '#b45309']

# Always convert to DataFrame with named columns
_Xsm_m = pd.DataFrame(X_sm, columns=E6_MULTI)
_Xlc_m = pd.DataFrame(X_lc, columns=E6_MULTI)

np.random.seed(42)
_Xsm_ms = _Xsm_m.sample(min(600, len(_Xsm_m)), random_state=42)
_Xlc_ms = _Xlc_m.sample(min(400, len(_Xlc_m)), random_state=42)

print("Computing SHAP — SmallMed model (q_sm_m) …")
_exp_sm_m = shap.TreeExplainer(q_sm_m)
_sv_sm_m  = _exp_sm_m.shap_values(_Xsm_ms)

print("Computing SHAP — LargeColossal model (q_lc_m) …")
_exp_lc_m = shap.TreeExplainer(q_lc_m)
_sv_lc_m  = _exp_lc_m.shap_values(_Xlc_ms)

# ── Plot 1: Global importance — both segments ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle("SHAP — Importancia Global de Features (|SHAP| medio) — Modelo Mensual",
             fontsize=13, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(_sv_sm_m, _Xsm_ms, feature_names=E6_MULTI, plot_type='bar',
                  show=False, max_display=len(E6_MULTI), color=_THESIS_SM)
axes[0].set_title("Segmento SmallMed (H+1..H+60)", fontweight='bold')
axes[0].set_facecolor('white')

plt.sca(axes[1])
shap.summary_plot(_sv_lc_m, _Xlc_ms, feature_names=E6_MULTI, plot_type='bar',
                  show=False, max_display=len(E6_MULTI), color=_THESIS_LC)
axes[1].set_title("Segmento LargeColossal (H+1..H+60)", fontweight='bold')
axes[1].set_facecolor('white')

plt.tight_layout()
plt.savefig('outputs_best/shap_monthly_importance.png', dpi=200)
plt.show()
print("Saved → outputs_best/shap_monthly_importance.png")

# ── Plot 2: Beeswarm — direction of effects (SmallMed) ───────────────────────
fig = plt.figure(figsize=(10, 7))
plt.title("SmallMed — Dirección de Efectos SHAP (beeswarm)", fontsize=12, fontweight='bold')
shap.summary_plot(_sv_sm_m, _Xsm_ms, feature_names=E6_MULTI, show=False,
                  max_display=len(E6_MULTI))
ax = plt.gca()
ax.set_facecolor('white')
plt.tight_layout()
plt.savefig('outputs_best/shap_monthly_beeswarm_sm.png', dpi=200)
plt.show()
print("Saved → outputs_best/shap_monthly_beeswarm_sm.png")

# ── Plot 3: SHAP by horizon bucket ───────────────────────────────────────────
_df_ms = _Xsm_ms.copy()
_buckets_m = {'H+1–12': (1, 12), 'H+13–24': (13, 24), 'H+25–36': (25, 36), 'H+37–60': (37, 60)}
_bucket_shap_m = {}
for bname, (lo, hi) in _buckets_m.items():
    mask = (_df_ms['Horizonte_feat'] >= lo) & (_df_ms['Horizonte_feat'] <= hi)
    if mask.sum() >= 5:
        _bucket_shap_m[bname] = np.abs(_sv_sm_m[mask.values]).mean(axis=0)

if _bucket_shap_m:
    fig, ax = plt.subplots(figsize=(14, 5))
    x = np.arange(len(E6_MULTI)); w = 0.2
    for i, (bname, vals) in enumerate(_bucket_shap_m.items()):
        ax.bar(x + i*w, vals, w, label=bname,
               color=_BUCKET_COLORS[i], alpha=0.85, edgecolor='white')
    ax.set_xticks(x + w*1.5)
    ax.set_xticklabels(E6_MULTI, rotation=35, ha='right', fontsize=8.5)
    ax.set_ylabel('|SHAP| medio')
    ax.set_title('SmallMed — Importancia SHAP por Horizonte Mensual', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('outputs_best/shap_monthly_by_horizon.png', dpi=200)
    plt.show()
    print("Saved → outputs_best/shap_monthly_by_horizon.png")

# ── Printed table ─────────────────────────────────────────────────────────────
_msm_m = np.abs(_sv_sm_m).mean(axis=0)
_mlc_m = np.abs(_sv_lc_m).mean(axis=0)
print("\n" + "="*68)
print(f"  {'Feature':25s}  {'|SHAP| SM':10s}  {'|SHAP| LC':10s}  Bar (SM)")
print("  " + "-"*58)
for i in np.argsort(_msm_m)[::-1]:
    bar = '█' * max(1, int(_msm_m[i]/_msm_m.max()*20))
    print(f"  {E6_MULTI[i]:25s}  {_msm_m[i]:.4f}      {_mlc_m[i]:.4f}      {bar}")

print("\nInterpretación clave (mensual):")
print("   Prod_Lag1 / Prod_Lag12 → memoria a 1 y 12 meses (estacionalidad)")
print("   Month_sin / Month_cos  → patrón estacional dentro del año")
print("   Prod_MA12              → tendencia reciente suavizada")
print("   Horizonte_feat         → distingue predicciones a 1 mes vs 5 años")
print("   Is_Pandemic_Orig/Target → corrección COVID-19 (2020-2021)")

Computing SHAP — SmallMed model (q_sm_m) …
Computing SHAP — LargeColossal model (q_lc_m) …


Saved → outputs_best/shap_monthly_importance.png


Saved → outputs_best/shap_monthly_beeswarm_sm.png
Saved → outputs_best/shap_monthly_by_horizon.png

  Feature                    |SHAP| SM   |SHAP| LC   Bar (SM)
  ----------------------------------------------------------
  Mine_Size                  0.1372      0.0438      ████████████████████
  Prod_pct_change_m          0.0812      0.1212      ███████████
  Horizonte_feat             0.0603      0.0478      ████████
  Company_Size               0.0530      0.0112      ███████
  Prod_Lag1                  0.0494      0.0660      ███████
  Prod_vs_peak               0.0486      0.0272      ███████
  Mine_age                   0.0393      0.0196      █████
  Prod_MA12                  0.0310      0.0355      ████
  Prod_pct_change_36m        0.0298      0.0506      ████
  Mine_share                 0.0273      0.0196      ███
  Mine_trend_slope_36m       0.0184      0.0177      ██
  Month_sin                  0.0103      0.0116      █
  Is_Pandemic_Target         0.0085      0.0165   

## Sección 11b — Resumen Comparativo de Modelos y Análisis por Mina

Compara todos los modelos evaluados y profundiza en el desempeño individual por mina:
- **Tabla resumen**: todos los experimentos ordenados por Hfoc% (H+36/48/60) dentro de cada segmento
- **WR por horizonte**: top-5 vs bottom-5 minas — evolución de 1 a 60 meses
- **Caso de estudio**: la mejor mina analizada en detalle (pred vs real, WR por horizonte, consistencia entre orígenes)


In [19]:

# ══════════════════════════════════════════════════════════════════════════════
# SECCIÓN 11b — Resumen comparativo de modelos + Análisis por mina (Mensual)
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

# ── Load predictions CSV (prefer FULL file with ensembles) ───────────────────
_path_full = 'outputs_best/predicciones_mensuales_baseline_completo.csv'
_path_ml   = 'outputs_best/predicciones_mensuales_baseline_extendido.csv'

if os.path.exists(_path_full):
    _dfm = pd.read_csv(_path_full)
    print(f"Loaded: {_path_full}")
elif os.path.exists(_path_ml):
    _dfm = pd.read_csv(_path_ml)
    print(f"Loaded: {_path_ml}")
else:
    raise FileNotFoundError('No monthly predictions file found in outputs_best/')

_dfm['Mine']   = _dfm['Mine'].str.lower().str.strip()
_dfm['Origin'] = pd.to_datetime(_dfm['Origin'], format='mixed')
HF_M = [36, 48, 60]

def _statsm(df, exp, hf=HF_M):
    s = df[df['Exp']==exp].copy()
    if len(s)==0: return None
    s = s[(s['Actual']>0)&(s['Pred']>0)].copy()
    s['ae']  = abs(s['Actual']-s['Pred'])
    s['aen'] = abs(s['Actual']-s['Naive_Pred'])
    s['ape'] = s['ae']/s['Actual']*100
    s['b']   = (s['ae']<s['aen']).astype(int)
    wr   = s['b'].mean()*100
    foc  = s[s['Horizonte'].isin(hf)]['b'].mean()*100
    mase = s['ae'].mean()/s['aen'].mean()
    sk   = (1-mase)*100
    m50  = s.groupby('Mine')['b'].mean().ge(0.5).sum(); n=s['Mine'].nunique()
    sizes= s['Mine_Size'].dropna().unique()
    seg  = 'SM' if len(sizes)>0 and max(sizes)<=1 else ('LC' if len(sizes)>0 and min(sizes)>=2 else 'All')
    return {'Modelo':exp, 'WR%':round(wr,1), 'Hfoc%':round(foc,1),
            'MASE':round(mase,3), 'MdAPE%':round(s['ape'].median(),1),
            'Skill%':round(sk,1), 'Minas≥50':f"{m50}/{n}", 'Seg':seg}

EXPS_M = [
    'LGB_LogRatio','LGB_MultiH','RF_MultiH','Ens_3070',
    'LGB_LargeCol','XGB_LargeCol','LGB_MultiH_LC','RF_MultiH_LC','Ens_LC_3070',
    'Ens_Segmentado','Ens_Super_Monthly',
]
_rows_m = [r for r in (_statsm(_dfm,e) for e in EXPS_M) if r]
_sbm = pd.DataFrame(_rows_m)

print("="*90)
print("RESUMEN COMPARATIVO — MODELOS MENSUALES  (H_focus = H+36, H+48, H+60)")
print("="*90)
_sbm['_ord'] = _sbm['Seg'].map({'SM':0,'LC':1,'All':2}).fillna(3)
_sbm = _sbm.sort_values(['_ord','Hfoc%'], ascending=[True,False]).drop(columns='_ord')
print(_sbm[['Modelo','Seg','WR%','Hfoc%','MASE','MdAPE%','Skill%','Minas≥50']].to_string(index=False))

best_sm_m = _sbm[_sbm['Seg']=='SM'].iloc[0]['Modelo'] if len(_sbm[_sbm['Seg']=='SM'])>0 else 'Ens_3070'
best_lc_m = _sbm[_sbm['Seg']=='LC'].iloc[0]['Modelo'] if len(_sbm[_sbm['Seg']=='LC'])>0 else 'LGB_MultiH_LC'
best_all_m= _sbm[_sbm['Seg']=='All'].iloc[0]['Modelo'] if len(_sbm[_sbm['Seg']=='All'])>0 else 'Ens_Segmentado'
print(f"\n✅ Mejor SmallMed: {best_sm_m}  |  Mejor LargeCol: {best_lc_m}  |  Mejor global: {best_all_m}")

# ── Per-mine ranking ──────────────────────────────────────────────────────────
_base_exp_m = 'Ens_Segmentado' if 'Ens_Segmentado' in _dfm['Exp'].values else ('Ens_3070' if 'Ens_3070' in _dfm['Exp'].values else _dfm['Exp'].iloc[0])
if _base_exp_m != 'Ens_Segmentado':
    print(f"⚠️ Ens_Segmentado no encontrado. Usando {_base_exp_m} para análisis por mina.")

_ensm = _dfm[(_dfm['Exp']==_base_exp_m)&(_dfm['Actual']>0)&(_dfm['Pred']>0)].copy()
_ensm['ae']  = abs(_ensm['Actual']-_ensm['Pred'])
_ensm['aen'] = abs(_ensm['Actual']-_ensm['Naive_Pred'])
_ensm['ape'] = _ensm['ae']/_ensm['Actual']*100
_ensm['b']   = (_ensm['ae']<_ensm['aen']).astype(int)

if _ensm.empty:
    raise ValueError('No hay datos válidos para análisis por mina (Actual/Pred > 0).')

_ms_m = _ensm.groupby('Mine').apply(lambda g: pd.Series({
    'WR%':    round(g['b'].mean()*100,1),
    'Hfoc%':  round(g[g['Horizonte'].isin(HF_M)]['b'].mean()*100,1),
    'MASE':   round(g['ae'].mean()/g['aen'].mean(),3),
    'MdAPE%': round(g['ape'].median(),1),
    'n':      len(g),
})).sort_values('WR%', ascending=False).reset_index()

try:
    _sbcsv_m = pd.read_csv('outputs_best/scoreboard_monthly_v7.csv')
    _sbcsv_m['Mine'] = _sbcsv_m['Mine'].str.lower().str.strip()
    _size_map_m = _sbcsv_m.set_index('Mine')['Size_Label'].to_dict()
    _ms_m['Size'] = _ms_m['Mine'].map(_size_map_m).fillna('?')
except: _ms_m['Size'] = '?'

print("\n" + "="*70)
print(f"RANKING POR MINA — {_base_exp_m} Mensual (H+1..H+60, todos los orígenes)")
print("="*70)
print(_ms_m[['Mine','Size','WR%','Hfoc%','MASE','MdAPE%','n']].to_string(index=False))

# ── Fig 1: WR by horizon — top 5 vs bottom 5 ─────────────────────────────────
top5_m = _ms_m.head(5)['Mine'].tolist()
bot5_m = _ms_m.tail(5)['Mine'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Win Rate por Horizonte — Ens_Segmentado Mensual", fontsize=13,
             fontweight='bold', color='white')
fig.patch.set_facecolor('#1e293b')
for ax in axes: ax.set_facecolor('#0f172a')

c_top = ['#22c55e','#4ade80','#34d399','#86efac','#6ee7b7']
c_bot = ['#ef4444','#f87171','#f97316','#fb923c','#fbbf24']

for mines, colors, title, ax in [
    (top5_m, c_top, 'Top 5 Minas (mejores)',           axes[0]),
    (bot5_m, c_bot, 'Bottom 5 Minas (más difíciles)',  axes[1]),
]:
    for j, mine in enumerate(mines):
        sg  = _ensm[_ensm['Mine']==mine]
        wrh = sg.groupby('Horizonte')['b'].mean()*100
        # Smooth: plot every 3rd horizon to avoid clutter
        wrh_s = wrh.iloc[::3]
        ax.plot(wrh_s.index, wrh_s.values, '-', label=mine.title(),
                color=colors[j], linewidth=2, alpha=0.9)
    ax.axhline(50, color='#94a3b8', linestyle='--', alpha=0.6, linewidth=1.2)
    for hf in HF_M:
        ax.axvline(hf, color='#f59e0b', linestyle=':', alpha=0.4, linewidth=1)
    ax.set_xlabel('Horizonte (meses)', color='#94a3b8')
    ax.set_ylabel('Win Rate %', color='#94a3b8')
    ax.set_title(title, color='white', fontsize=11)
    ax.tick_params(colors='#94a3b8'); ax.grid(alpha=0.2)
    [sp.set_color('#334155') for sp in ax.spines.values()]
    ax.legend(fontsize=8.5)

axes[0].annotate('H+36/48/60\n(focus)', xy=(36,52), color='#f59e0b', fontsize=8)
plt.tight_layout()
plt.savefig('outputs_best/wr_by_horizon_mines_monthly.png', dpi=150, bbox_inches='tight',
            facecolor='#1e293b')
plt.show()
print("Saved → outputs_best/wr_by_horizon_mines_monthly.png")

# ── Fig 2: Best mine case study ───────────────────────────────────────────────
best_mine_m = _ms_m.iloc[0]['Mine']
bm_wr_m     = _ms_m.iloc[0]['WR%']
_bm_m = _ensm[_ensm['Mine']==best_mine_m].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Caso de Estudio: {best_mine_m.title()}  (WR={bm_wr_m}%, MASE={_ms_m.iloc[0]['MASE']})",
             fontsize=13, fontweight='bold', color='white')
fig.patch.set_facecolor('#1e293b')
for ax in axes: ax.set_facecolor('#0f172a')

# Scatter pred vs actual (colour = horizon bucket)
_hbkt = pd.cut(_bm_m['Horizonte'], bins=[0,12,24,36,60], labels=['H1-12','H13-24','H25-36','H37-60'])
_palette = {'H1-12':'#3b82f6','H13-24':'#f59e0b','H25-36':'#f97316','H37-60':'#22c55e'}
for bkt, grp in _bm_m.groupby(_hbkt):
    axes[0].scatter(grp['Actual'], grp['Pred'], c=_palette.get(str(bkt),'#94a3b8'),
                    alpha=0.65, s=20, label=str(bkt), edgecolors='none')
_mn = min(_bm_m['Actual'].min(),_bm_m['Pred'].min())*0.88
_mx = max(_bm_m['Actual'].max(),_bm_m['Pred'].max())*1.08
axes[0].plot([_mn,_mx],[_mn,_mx],'--',color='#94a3b8',linewidth=1.2)
axes[0].legend(fontsize=7.5); axes[0].set_title('Pred vs Real', color='white')
axes[0].set_xlabel('Real (kt/mes)', color='#94a3b8'); axes[0].set_ylabel('Predicción', color='#94a3b8')
axes[0].tick_params(colors='#94a3b8'); axes[0].grid(alpha=0.2)

# WR by horizon (smoothed)
_wrh_m = _bm_m.groupby('Horizonte')['b'].mean()*100
axes[1].fill_between(_wrh_m.index, _wrh_m.values,
                     color='#22c55e', alpha=0.35, label='WR%')
axes[1].plot(_wrh_m.index, _wrh_m.values, color='#22c55e', linewidth=1.5)
axes[1].axhline(50, color='#f59e0b', linestyle='--', linewidth=1.2)
for hf in HF_M:
    axes[1].axvline(hf, color='#f59e0b', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Horizonte (meses)', color='#94a3b8'); axes[1].set_ylabel('Win Rate %', color='#94a3b8')
axes[1].set_title('Win Rate por Horizonte', color='white')
axes[1].tick_params(colors='#94a3b8'); axes[1].grid(alpha=0.2)

# Per-origin WR
_per_orig_m = _bm_m.groupby(_bm_m['Origin'].dt.strftime('%Y-%m'))['b'].mean()*100
axes[2].bar(_per_orig_m.index, _per_orig_m.values,
            color=['#22c55e' if v>=50 else '#ef4444' for v in _per_orig_m.values],
            alpha=0.85, edgecolor='#1e293b')
axes[2].axhline(50, color='#f59e0b', linestyle='--', linewidth=1.2)
axes[2].set_xlabel('Origen', color='#94a3b8'); axes[2].set_ylabel('Win Rate %', color='#94a3b8')
axes[2].set_title('Win Rate por Origen', color='white')
axes[2].tick_params(axis='x', rotation=45, colors='#94a3b8'); axes[2].tick_params(axis='y', colors='#94a3b8')
axes[2].grid(axis='y', alpha=0.2)
[sp.set_color('#334155') for ax in axes for sp in ax.spines.values()]

plt.tight_layout()
plt.savefig(f"outputs_best/case_study_{best_mine_m.replace(' ','_')}_monthly.png",
            dpi=150, bbox_inches='tight', facecolor='#1e293b')
plt.show()
print(f"Saved → outputs_best/case_study_{best_mine_m.replace(' ','_')}_monthly.png")

# ── Top-3 mines: focus horizon detail table ───────────────────────────────────
print(f"\nTop 3 minas — detalle H+36/H+48/H+60:")
for mine in _ms_m.head(3)['Mine']:
    sg = _ensm[_ensm['Mine']==mine]
    row_str = f"  {mine.title():25s}"
    for hf in HF_M:
        sg_h = sg[sg['Horizonte']==hf]
        wr_h = sg_h['b'].mean()*100 if len(sg_h)>0 else float('nan')
        row_str += f"  H{hf}={wr_h:.0f}%"
    print(row_str)


Loaded: outputs_best/predicciones_mensuales_baseline_completo.csv
RESUMEN COMPARATIVO — MODELOS MENSUALES  (H_focus = H+36, H+48, H+60)
        Modelo Seg  WR%  Hfoc%  MASE  MdAPE%  Skill% Minas≥50
      Ens_3070  SM 55.9   54.7 0.971    14.9     2.9    10/16
    LGB_MultiH  SM 53.0   54.5 0.978    16.0     2.2    12/16
     RF_MultiH  SM 50.3   50.7 0.990    16.4     1.0     7/16
  LGB_LogRatio  SM 50.9   34.4 0.991    14.1     0.9     7/16
   Ens_LC_3070  LC 58.0   52.6 0.920    15.8     8.0    14/16
  RF_MultiH_LC  LC 54.8   52.4 0.986    17.1     1.4    13/16
  XGB_LargeCol  LC 51.5   51.1 1.054    17.5    -5.4     9/16
 LGB_MultiH_LC  LC 52.4   48.3 1.018    17.4    -1.8     9/16
  LGB_LargeCol  LC 50.1   45.2 1.036    17.2    -3.6     8/16
Ens_Segmentado All 56.8   54.1 0.930    15.3     7.0    22/28

✅ Mejor SmallMed: Ens_3070  |  Mejor LargeCol: Ens_LC_3070  |  Mejor global: Ens_Segmentado

RANKING POR MINA — Ens_Segmentado Mensual (H+1..H+60, todos los orígenes)
            Mi

Saved → outputs_best/wr_by_horizon_mines_monthly.png
Saved → outputs_best/case_study_atacama_kozan_monthly.png

Top 3 minas — detalle H+36/H+48/H+60:
  Atacama Kozan              H36=70%  H48=56%  H60=71%
  Antucoya                   H36=78%  H48=75%  H60=83%
  Collahuasi                 H36=83%  H48=67%  H60=nan%


In [20]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ─── Thesis style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.size': 10, 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
})
_BLUE  = '#1d4ed8'
_TEAL  = '#0d9488'
_GREEN = '#16a34a'
_RED   = '#dc2626'

# ── Cross-model: Mensual vs Anual ─────────────────────────────────────────────
_ann_path = os.path.normpath(os.path.join(EXPORT_DIR, '../../annual_model/outputs_best/scoreboard_annual_v7.csv'))
try:
    _df_ann = pd.read_csv(_ann_path)
    _df_mon = mine_agg.copy()
    _df_ann['_k'] = _df_ann['Mine'].str.lower().str.strip()
    _df_mon['_k'] = _df_mon['Mine'].str.lower().str.strip()

    # Build merge — handle MdAPE_Skill only if present in both
    _ann_cols = ['_k', 'Skill', 'WR', 'MASE']
    _mon_cols = ['_k', 'Mine', 'Skill', 'WR', 'MASE']
    if 'MdAPE_Skill' in _df_ann.columns: _ann_cols += ['MdAPE_Skill']
    if 'MdAPE_Skill' in _df_mon.columns: _mon_cols += ['MdAPE_Skill']
    if 'WSkill' in _df_ann.columns: _ann_cols += ['WSkill']
    if 'WSkill' in _df_mon.columns: _mon_cols += ['WSkill']

    _cross = _df_mon[_mon_cols].merge(
        _df_ann[_ann_cols].rename(columns={c: f'{c}_A' for c in _ann_cols if c != '_k'}),
        on='_k', how='inner').drop(columns='_k')
    _cross['ΔSkill'] = (_cross['Skill'] - _cross['Skill_A']).round(1)
    _cross['ΔWR']    = ((_cross['WR'] - _cross['WR_A']) * 100).round(1)
    _cross = _cross.sort_values('ΔSkill', ascending=False)
    _cross.to_csv(f'{EXPORT_DIR}/cross_model_comparison.csv', index=False)

    print("Cross-model: Mensual vs Anual (Ens_Segmentado)")
    print(f"  {'Mine':<35} {'Skill_M':>8} {'Skill_A':>8} {'ΔSkill':>7} {'WR_M%':>7} {'WR_A%':>7} {'ΔWR':>7}")
    print("  " + "-"*83)
    for _, r in _cross.iterrows():
        em = '>>>' if abs(r['ΔSkill']) > 10 else '   '
        wr_m = r['WR'] * 100 if r['WR'] <= 1.5 else r['WR']
        wr_a = r['WR_A'] * 100 if r['WR_A'] <= 1.5 else r['WR_A']
        print(f"  {em} {r['Mine']:<31} {r['Skill']:>+8.1f} {r['Skill_A']:>+8.1f} {r['ΔSkill']:>+7.1f}"
              f" {wr_m:>7.1f} {wr_a:>7.1f} {r['ΔWR']:>+7.1f}")
    m_w = (_cross['ΔSkill'] > 0).sum()
    a_w = (_cross['ΔSkill'] < 0).sum()
    print(f"\n  Mensual mejor: {m_w}/{len(_cross)} | Anual mejor: {a_w}/{len(_cross)}")

    # ── Plot ──────────────────────────────────────────────────────────────────
    _cross_s = _cross.sort_values('ΔSkill', ascending=True)
    n = len(_cross_s); y = range(n); bh = 0.38

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, max(6, n * 0.42)))

    # WR
    ax1.barh([i + bh/2 for i in y], [r['WR']*100 if r['WR']<=1.5 else r['WR']
              for _, r in _cross_s.iterrows()],
             height=bh, color=_TEAL,  alpha=0.85, label='Mensual')
    ax1.barh([i - bh/2 for i in y], [r['WR_A']*100 if r['WR_A']<=1.5 else r['WR_A']
              for _, r in _cross_s.iterrows()],
             height=bh, color=_THESIS_SM,  alpha=0.85, label='Anual')
    ax1.axvline(50, color='#6b7280', ls='--', lw=1.2)
    ax1.set_yticks(list(y))
    ax1.set_yticklabels([m.replace('_', ' ').title() for m in _cross_s['Mine']], fontsize=8)
    ax1.set_xlabel('Win Rate (%)'); ax1.set_title('Win Rate: Anual vs Mensual', fontweight='bold')
    ax1.legend()

    # Skill
    ax2.barh([i + bh/2 for i in y], _cross_s['Skill'],   height=bh,
             color=_TEAL,  alpha=0.85, label='Mensual')
    ax2.barh([i - bh/2 for i in y], _cross_s['Skill_A'], height=bh,
             color=_THESIS_SM,  alpha=0.85, label='Anual')
    ax2.axvline(0, color='#6b7280', ls='--', lw=1.2)
    ax2.set_yticks(list(y)); ax2.set_yticklabels([])
    ax2.set_xlabel('Skill Score MAE (%)'); ax2.set_title('Skill Score: Anual vs Mensual', fontweight='bold')
    ax2.legend()

    fig.suptitle('Comparación Cross-Model por Mina — Anual vs Mensual (Ens. Segmentado)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{EXPORT_DIR}/cross_model_comparison.png', dpi=200)
    plt.show()
    print(f"Saved → {EXPORT_DIR}/cross_model_comparison.png")

except FileNotFoundError:
    print(f"Annual scoreboard no encontrado en: {_ann_path}\nCorrer modelo anual primero.")

Annual scoreboard no encontrado en: ../annual_model/outputs_best/scoreboard_annual_v7.csv
Correr modelo anual primero.

In [21]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

# ─── Thesis style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e5e7eb', 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#374151', 'xtick.color': '#374151', 'ytick.color': '#374151',
    'text.color': '#111827', 'font.size': 10, 'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
})
_BLUE = '#1d4ed8'; _NAVY = '#1e3a5f'; _GRAY = '#6b7280'

# ── Proyecciones mensuales 2026-2032 con bandas CI ───────────────────────────
_df_proj_m = pd.read_csv(f'{EXPORT_DIR}/proyecciones_mensuales_2026_2032.csv')
_df_proj_m['ForecastDate'] = pd.to_datetime(_df_proj_m['ForecastDate'])
_df_proj_m = _df_proj_m[_df_proj_m['Horizonte'] <= 60]  # Limitar a H≤60

_hist_m = df_raw[df_raw['Date'] >= pd.Timestamp('2023-01-01')][['Match_Key', 'Date', 'Production']].copy()
_hist_m.columns = ['Mine', 'Date', 'Production']

top_mines_m = mine_agg.nlargest(12, 'WR')['Mine'].tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
fig.suptitle('Proyecciones Mensuales 2026–2032 — Ensamble Segmentado\n(base ± IC q10–q90, top 12 por WR)',
             fontsize=13, fontweight='bold')

for idx, mine in enumerate(top_mines_m[:12]):
    ax = axes[idx // 4][idx % 4]
    h_m = _hist_m[_hist_m['Mine'] == mine].sort_values('Date')
    p_m = _df_proj_m[_df_proj_m['Mine'] == mine].sort_values('ForecastDate')
    if not h_m.empty:
        ax.plot(h_m['Date'], h_m['Production'],
                color=_NAVY, lw=1.5, alpha=0.8, label='Histórico')
    if not p_m.empty:
        ax.plot(p_m['ForecastDate'], p_m['Pred'],
                color=_THESIS_SM, lw=1.8, label='Pronóstico')
        ax.fill_between(p_m['ForecastDate'], p_m['Lower'], p_m['Upper'],
                        color=_THESIS_SM, alpha=0.15, label='IC 80 %')
        ax.axhline(p_m['Naive_Pred'].iloc[0], color=_GRAY, lw=1.2, ls=':', label='Naïve')
    wr_v = mine_agg[mine_agg['Mine'] == mine]['WR'].values
    sk_v = mine_agg[mine_agg['Mine'] == mine]['Skill'].values
    mine_disp = (mine.replace('_', ' ').title())
    wr_str = f' WR={wr_v[0]*100:.0f}%' if len(wr_v) else ''
    sk_str = f' Sk={sk_v[0]:+.0f}%'   if len(sk_v) else ''
    ax.set_title(f'{mine_disp}\n{wr_str}{sk_str}', fontsize=8.5, fontweight='bold')
    ax.set_ylabel('kt Cu', fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'))
    ax.tick_params(axis='both', labelsize=7)
    ax.tick_params(axis='x', rotation=30)
    if idx == 0:
        ax.legend(fontsize=7)

# Global legend
handles = [
    plt.Line2D([0], [0], color=_NAVY, lw=1.5, alpha=0.8, label='Histórico'),
    plt.Line2D([0], [0], color=_THESIS_SM, lw=1.8,             label='Pronóstico'),
    Patch(facecolor=_THESIS_SM, alpha=0.2, label='IC 80 %'),
    plt.Line2D([0], [0], color=_GRAY, ls=':', lw=1.2, label='Naïve'),
]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.savefig(f'{EXPORT_DIR}/projections_plot_monthly.png', dpi=200)
plt.show()
print(f"Saved → {EXPORT_DIR}/projections_plot_monthly.png")

Saved → outputs_best/projections_plot_monthly.png
